In [3]:
import os
import platform
import sys
import multiprocessing

def get_ram_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / (1024**3), 2)
    except ImportError:
        return "psutil not installed"

print("=== SYSTEM INFORMATION ===")
print(f"OS: {platform.system()} {platform.release()}")
print(f"OS Version: {platform.version()}")
print(f"Machine: {platform.machine()}")
print(f"Processor: {platform.processor()}")
print(f"CPU Cores (logical): {multiprocessing.cpu_count()}")
print(f"Python Version: {sys.version}")
print(f"RAM (GB): {get_ram_gb()}")

=== SYSTEM INFORMATION ===
OS: Linux 6.8.12-11-pve
OS Version: #1 SMP PREEMPT_DYNAMIC PMX 6.8.12-11 (2025-05-22T09:39Z)
Machine: x86_64
Processor: x86_64
CPU Cores (logical): 8
Python Version: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
RAM (GB): 32.0


In [4]:
print("\n=== GPU INFORMATION ===")

try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"GPU Count: {torch.cuda.device_count()}")
        print(f"GPU Total Memory (GB): {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)}")
        print(f"CUDA Version (PyTorch): {torch.version.cuda}")
        print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")
except ImportError:
    print("PyTorch not installed")


=== GPU INFORMATION ===
PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU Name: NVIDIA A100-PCIE-40GB
GPU Count: 1
GPU Total Memory (GB): 39.49
CUDA Version (PyTorch): 12.8
GPU Compute Capability: (8, 0)


# Original Jeepney Routes

## Route Connection and Genetic Algorithm Functions

### Setting Up

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import geopandas as gpd
import networkx as nx
import shapely
import folium
import geojson
import math
import osmnx as ox
from rtree import index as rtree_index
import pickle
import copy
import utm
import csv
from pyproj import Transformer
from shapely.ops import transform

from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon, LineString, Point
from geopy.distance import geodesic
from shapely.ops import split

from __future__ import absolute_import, division
from math import radians, sin, cos, sqrt, atan2, exp, log
import webbrowser
import random
from scipy.spatial import KDTree
from scipy.spatial.distance import euclidean

import os
import time
from PIL import Image, ImageChops
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

ox.settings.log_console=True
ox.settings.use_cache=True

# Connectivity metric configuration
TARGET_NODE_RATIO = 0.75   # for N_i
CITY_CONNECTIVITY_CONFIG = {
    "Manila": {
        "min_intersections_per_route": 4,
        "min_transpo_stops_per_route": 3,
    },
    "Makati": {
        "min_intersections_per_route": 3,
        "min_transpo_stops_per_route": 2,
    },
    "Mandaluyong": {
        "min_intersections_per_route": 2,
        "min_transpo_stops_per_route": 1,
    },
}

# Total potential transport nodes. Set them equal to corresponding GA values
total_potential_transport_nodes_Manila = 464
total_potential_transport_nodes_Makati = 111
total_potential_transport_nodes_Mandaluyong = 45
# Population-demand normalization
POPULATION_DENOMINATOR_PERCENTILE = 95

In [6]:
# For reproducibility. Makes any random sampling deterministic.
SEED = 42 # You can change this value to any integer depending on the run
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); 
np.random.seed(SEED)

In [7]:
# Defining classes for the dataframes      
class stopCandidate:
    def __init__(self, lat, long, isTranspo, id, area, zone_ids=None, source_edge_id=None, transport_source_id=None):
        self.lat = lat
        self.long = long
        self.isTranspo = isTranspo
        self.enabled = False
        self.id = id
        self.area = area
        self.degree = 0
        # Spatial provenance
        self.zone_ids = (
            list(zone_ids)
            if zone_ids
            else []
        )
        self.source_edge_id = source_edge_id
        # Population-demand attributes
        self.population_raw = 0.0
        self.population_score = 0.0
        # T_n
        self.intermodal_indicator = (
            1 if isTranspo else 0
        )
        # Flood attributes
        self.flood_5 = 0
        self.flood_25 = 0
        self.flood_100 = 0
        self.flood_score = 0.0
        self.transport_source_id = transport_source_id
        
    def getLat(self):
        return self.lat
    
    def getLong(self):
        return self.long
    
    def getDegree(self):
        return self.degree
    
class networkObj:
    def __init__(
        self,
        route_network,
        graph,
        reversed_routes=None
    ):
        self.routes = route_network
        self.graph = graph
        self.reversed_routes = (
            list(reversed_routes)
            if reversed_routes is not None
            else []
        )

        self.fitness_score = 0.0
        self.fitness_score_components = {}
        
class Route:
    def __init__(self, route, route_id, length):
        self.route = route
        self.route_id = route_id
        self.length = length

In [8]:
# For units
def degrees_to_meters(angle_degrees):
    return angle_degrees * 6371000 * math.pi / 180

def meters_to_degrees(distance_meters):
    return distance_meters / 6371000 * 180 / math.pi

In [9]:
# Import and Export networks or graphs to pickle
def export_networks(networks, path):
    with open(path, 'wb') as f:
        pickle.dump(networks, f)

def import_networks(path):
    with open(path, 'rb') as f:
        routes = pickle.load(f)
    return routes

In [10]:
import re

def process_csv(file_path):
    with open(file_path, mode='r', newline='') as file:
        point_pattern = re.compile(r'POINT \(([-\d.]+) ([-\d.]+)\)')
        reader = csv.DictReader(file)
        
        # Variables to hold the current route list and all routes list
        current_route_number = None
        current_route_list = []
        all_routes_list = []

        for row in reader:
            wkt = row['WKT']
            route_number = row['Route Number']
            
            match = point_pattern.match(wkt)
            if match:
                x, y = match.groups()
                x, y = float(x), float(y)
            else:
                x, y = None, None

            # Check if the route number has changed
            if route_number != current_route_number:
                # If there's a current route list, append it to the all routes list
                if current_route_list:
                    all_routes_list.append(current_route_list)
                
                # Start a new route list
                current_route_list = []
                current_route_number = route_number

            # Append the current row to the current route list
            current_route_list.append([(x, y), route_number])

        # Append the last route list to the all routes list
        if current_route_list:
            all_routes_list.append(current_route_list)
        
    return all_routes_list

#### Metric-Geometry

In [11]:
# Coordinate reference systems
WGS84_CRS = "EPSG:4326"
METRIC_CRS = "EPSG:32651"  # WGS 84 / UTM Zone 51N

# Reusable coordinate transformers
TO_METRIC = Transformer.from_crs(
    WGS84_CRS,
    METRIC_CRS,
    always_xy=True
).transform

TO_WGS84 = Transformer.from_crs(
    METRIC_CRS,
    WGS84_CRS,
    always_xy=True
).transform


def to_metric_geometry(geometry):
    """
    Convert a WGS84 geometry to UTM Zone 51N.
    Coordinates in the returned geometry are in meters.
    """
    return transform(TO_METRIC, geometry)


def to_wgs84_geometry(geometry):
    """
    Convert a UTM Zone 51N geometry back to WGS84.
    """
    return transform(TO_WGS84, geometry)


def geometry_area_m2(geometry):
    """
    Return polygon area in square meters.
    """
    return to_metric_geometry(geometry).area


def geometry_perimeter_m(geometry):
    """
    Return geometry length/perimeter in meters.
    """
    return to_metric_geometry(geometry).length


def geometry_distance_m(geometry1, geometry2):
    """
    Return minimum distance between two geometries in meters.
    """
    geom1_metric = to_metric_geometry(geometry1)
    geom2_metric = to_metric_geometry(geometry2)

    return geom1_metric.distance(geom2_metric)


def buffer_geometry_m(geometry, distance_m):
    """
    Create a buffer of the specified number of meters while returning
    the resulting geometry in WGS84.
    """
    metric_geometry = to_metric_geometry(geometry)
    metric_buffer = metric_geometry.buffer(distance_m)

    return to_wgs84_geometry(metric_buffer)

#### Graph and Features

In [12]:
# GENERATION OF MAIN CITY GRAPH
# IF FIRST TIME RUNNING, RUN THIS CODE TO GENERATE THE GRAPH
def generate_graph():
    mode = 'drive'
    graph = ox.graph_from_place(select_city, network_type = mode) # Generate graph of Metro manila
    ox.save_graphml(graph, city_file) # Save it as a file

def load_graph():
    graph = ox.load_graphml(city_file)
    
    print("Graph loaded successfully")
    print("NUMBER OF EDGES: ", graph.number_of_edges())
    print("NUMBER OF NODES: ", graph.number_of_nodes())
    print('\n')
    return graph

select_city = "Metro Manila, Philippines"
city_file = 'map/Metro Manila.graphml'
#generate_graph()
METROMANILA_GRAPH = load_graph()

select_city = "Manila, Philippines"
city_file = 'map/Manila.graphml'
#generate_graph()
MANILA_GRAPH = load_graph()

select_city = "Mandaluyong, Philippines"
city_file = 'map/Mandaluyong.graphml'
#generate_graph()
MANDALUYONG_GRAPH = load_graph()

select_city = "Makati, Philippines"
city_file = 'map/Makati.graphml'
#generate_graph()
MAKATI_GRAPH = load_graph()

Graph loaded successfully
NUMBER OF EDGES:  149172
NUMBER OF NODES:  59386


Graph loaded successfully
NUMBER OF EDGES:  12739
NUMBER OF NODES:  5062


Graph loaded successfully
NUMBER OF EDGES:  2452
NUMBER OF NODES:  1024


Graph loaded successfully
NUMBER OF EDGES:  5558
NUMBER OF NODES:  2294




In [13]:
def assign_edge_ids(graph):
    for edge_id, (u, v, k) in enumerate(
        graph.edges(keys=True)
    ):
        graph[u][v][k]["edge_id"] = edge_id


assign_edge_ids(MANILA_GRAPH)
assign_edge_ids(MAKATI_GRAPH)
assign_edge_ids(MANDALUYONG_GRAPH)

In [14]:
### For Filtering the roads and other features
# GETTING ROADS AND WATERWAYS

# Get all the roads in Manila
manila_road = ox.graph_to_gdfs(MANILA_GRAPH, nodes=False, edges=True)
MANILA_EDGE_GEOMETRY_BY_ID = {
    int(row["edge_id"]): row["geometry"]
    for _, row in manila_road.iterrows()
    if pd.notna(row.get("edge_id"))
}
filtered_roads = manila_road[manila_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk']

# Get all the roads with the allowed road types
filtered_roads_strings_manila = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)] 

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MANILA = 0
for index, row in filtered_roads_strings_manila.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MANILA += row['length']
    
# MAKATI
makati_road = ox.graph_to_gdfs(MAKATI_GRAPH, nodes=False, edges=True)
MAKATI_EDGE_GEOMETRY_BY_ID = {
    int(row["edge_id"]): row["geometry"]
    for _, row in makati_road.iterrows()
    if pd.notna(row.get("edge_id"))
}
filtered_roads = makati_road[makati_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk']

# Get all the roads with the allowed road types
filtered_roads_strings_makati = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)] 

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MAKATI = 0
for index, row in filtered_roads_strings_makati.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MAKATI += row['length']


# MANDALUYONG
mandaluyong_road = ox.graph_to_gdfs(MANDALUYONG_GRAPH, nodes=False, edges=True)
MANDALUYONG_EDGE_GEOMETRY_BY_ID = {
    int(row["edge_id"]): row["geometry"]
    for _, row in mandaluyong_road.iterrows()
    if pd.notna(row.get("edge_id"))
}
filtered_roads = mandaluyong_road[mandaluyong_road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = ['primary', 'secondary', 'tertiary', 'trunk']

# Get all the roads with the allowed road types
filtered_roads_strings_mandaluyong = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)] 

# Total Distance of the allowed road types
TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG = 0
for index, row in filtered_roads_strings_mandaluyong.iterrows():
    TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG += row['length']

### Reading Data

In [15]:
# Reading all points

manila_points_gdf = gpd.read_file('./City Data/Manila City/Manila_point.geojson')
mandaluyong_points_gdf = gpd.read_file('./City Data/Mandaluyong City/Mandaluyong_point.geojson')
makati_points_gdf = gpd.read_file('./City Data/Makati City/Makati_point.geojson')

manila_transpo_gdf = manila_points_gdf[manila_points_gdf['amenity'] == 'transportation']
mandaluyong_transpo_gdf = mandaluyong_points_gdf[mandaluyong_points_gdf['amenity'] == 'transportation']
makati_transpo_gdf = makati_points_gdf[makati_points_gdf['amenity'] == 'transportation']

In [16]:
# Index for optimization
manila_idx = rtree_index.Index()
for i, row in manila_transpo_gdf.iterrows():
    manila_idx.insert(i, row['geometry'].bounds)
    
mandaluyong_idx = rtree_index.Index()
for i, row in mandaluyong_transpo_gdf.iterrows():
    mandaluyong_idx.insert(i, row['geometry'].bounds)
    
makati_idx = rtree_index.Index()
for i, row in makati_transpo_gdf.iterrows():
    makati_idx.insert(i, row['geometry'].bounds)
    

In [17]:
# Function to Extract all the stop points per row
def extract_points(df):
    routes = []
    columns = list(df.columns)
    
    for index, row in df.iterrows():
        points = []
        for col in columns:
            cell_value = row[col]
            if pd.isna(cell_value):
                continue
                
            if cell_value.endswith(']'):
                cell_value = cell_value[:-1] + ',' # Remove the last bracket if it's there
                
            data = eval(cell_value)[0]
            
            lat = data[0]
            lon = data[1]
            points.append((lat, lon))
        routes.append(points)
    return routes

In [18]:
# REVISED ROUTES

# MANILA
file_path = 'Original Jeepney Routes/Original Data/Formatted-Manila-Routes.csv'
manila_route_stops = process_csv(file_path)

# MAKATI
file_path = 'Original Jeepney Routes/Original Data/Existing Jeepney Routes (Makati).csv'
makati_route_stops = process_csv(file_path)

# MANDALUYONG
file_path = 'Original Jeepney Routes/Original Data/Existing Jeepney Routes (Mandaluyong).csv'
mandaluyong_route_stops = process_csv(file_path)

### Stop Placement

In [19]:
# Checks if point is out of the specific geographic location
def remove_if_out_of_bounds(city_graph, routes, tolerance):
    nodes = ox.graph_to_gdfs(city_graph, edges=False)
    min_x, min_y, max_x, max_y = nodes.total_bounds
    
    new_routes = []
    for route in routes:
        new_route = []
        for point,name in route:
            long = point[0]
            lat = point[1]
            
            if min_x <= long <= max_x and min_y <= lat <= max_y:
                # Convert the graph to GeoDataFrame for easier spatial operations
                nearest_edge, distance_to_edge = ox.distance.nearest_edges(city_graph, X=long, Y=lat, return_dist=True)
                
            
                # Return True if any distance is less than the tolerance
                if distance_to_edge < tolerance:
                    #print(distance_to_edge)
                    new_route.append([point, name])
                else:
                    break
        new_routes.append(new_route)
            
    return new_routes

In [20]:
# This function checks if there are stops that are transpo stop and labels them as such
def get_close_points(idx, point, max_distance=50):

    lat_buffer = max_distance / 111_320.0

    lon_buffer = max_distance / (
        111_320.0
        * max(
            0.01,
            math.cos(math.radians(point.y))
        )
    )

    return list(
        idx.intersection((
            point.x - lon_buffer,
            point.y - lat_buffer,
            point.x + lon_buffer,
            point.y + lat_buffer
        ))
    )
        

def check_transpo_stops(
    routes,
    point_gdf,
    city_idx,
    max_distance_to_stop,
    canonical_transport_ids=None
):
    transpo_points = []
    non_transpo_points = []
    new_route_list_with_stop_id = []

    stop_id = 0

    for route in routes:

        new_route = []

        for point, name in route:

            lon, lat = point

            isTranspo = False
            transport_source_id = None
            closest_distance = float("inf")

            close_points = get_close_points(
                city_idx,
                Point(lon, lat)
            )

            for i, row in (
                point_gdf.loc[close_points].iterrows()
            ):

                if row["amenity"] != "transportation":
                    continue

                distance_m = geodesic(
                    (lat, lon),
                    (row["y"], row["x"])
                ).meters

                if (
                    distance_m <= max_distance_to_stop
                    and distance_m < closest_distance
                ):
                    closest_distance = distance_m
                    isTranspo = True

                    if canonical_transport_ids is not None:

                        if i not in canonical_transport_ids:
                            raise ValueError(
                                f"Raw transportation POI {i} "
                                "has no matching cleaned "
                                "GA transportation source."
                            )

                        transport_source_id = (
                            canonical_transport_ids[i]
                        )

                    else:
                        transport_source_id = i

            new_stop = stopCandidate(
                lat,
                lon,
                isTranspo,
                stop_id,
                0.0,
                transport_source_id=transport_source_id
            )

            new_route.append(new_stop)

            if isTranspo:
                transpo_points.append(new_stop)
            else:
                non_transpo_points.append(new_stop)

            stop_id += 1

        new_route_list_with_stop_id.append(new_route)

    return (
        transpo_points,
        non_transpo_points,
        new_route_list_with_stop_id
    )

In [21]:
# x - lon - 120
# y - lat - 14
# LineString(lon, lat)
# geodesic(lat, lon)

from shapely.ops import substring


def add_point_to_graph(
    point,
    graph_of_stops,
    list_of_stops,
    is_transpo
):
    lat = point.lat
    long = point.long

    # Find the exact nearest MultiDiGraph edge
    nearest_edge = ox.distance.nearest_edges(
        graph_of_stops,
        X=long,
        Y=lat,
        return_dist=False
    )

    node1, node2, edge_key = nearest_edge

    # Retrieve the exact edge
    data = graph_of_stops.get_edge_data(
        node1,
        node2,
        edge_key
    )
    
    if data is None:
        raise ValueError(
            f"Edge ({node1}, {node2}, {edge_key}) "
            "was returned by nearest_edges() but could not be found."
        )
    
    source_edge_id = data.get("edge_id")

    if source_edge_id is not None:
        source_edge_id = int(source_edge_id)

    # Get road geometry
    if "geometry" in data:
        line = data["geometry"]
    else:
        line = LineString([
            (
                graph_of_stops.nodes[node1]["x"],
                graph_of_stops.nodes[node1]["y"]
            ),
            (
                graph_of_stops.nodes[node2]["x"],
                graph_of_stops.nodes[node2]["y"]
            )
        ])

    # -------------------------------------------------
    # Snap the geocoded stop to the selected road edge
    # -------------------------------------------------
    line_metric = to_metric_geometry(line)

    original_point_metric = to_metric_geometry(
        Point(long, lat)
    )

    position_m = line_metric.project(
        original_point_metric
    )

    snapped_point_metric = line_metric.interpolate(
        position_m
    )

    snapped_point = to_wgs84_geometry(
        snapped_point_metric
    )

    snapped_long = snapped_point.x
    snapped_lat = snapped_point.y

    # -------------------------------------------------
    # Split the actual edge geometry at the snapped point
    # -------------------------------------------------
    total_length_m = line_metric.length
    
    epsilon_m = 0.01

    if total_length_m <= 2 * epsilon_m:
        raise ValueError(
            "Selected road edge is too short to split."
        )

    position_m = min(
        max(position_m, epsilon_m),
        total_length_m - epsilon_m
    )

    snapped_point_metric = line_metric.interpolate(
        position_m
    )

    snapped_point = to_wgs84_geometry(
        snapped_point_metric
    )

    snapped_long = snapped_point.x
    snapped_lat = snapped_point.y

    line_ab_metric = substring(
        line_metric,
        0,
        position_m
    )

    line_bc_metric = substring(
        line_metric,
        position_m,
        total_length_m
    )

    line_ab = to_wgs84_geometry(
        line_ab_metric
    )

    line_bc = to_wgs84_geometry(
        line_bc_metric
    )
    
    point.source_edge_id = source_edge_id

    # -------------------------------------------------
    # Add the stop node
    # -------------------------------------------------
    graph_of_stops.add_node(
        point.id,
        x=snapped_long,
        y=snapped_lat,
        isTranspo=is_transpo,
        area=0.0,
        source_edge_id=source_edge_id,
        transport_source_id=(
            point.transport_source_id
        ),
        zone_ids=[]
    )

    # Keep the stopCandidate coordinates consistent
    # with the graph node coordinates.
    point.long = snapped_long
    point.lat = snapped_lat

    list_of_stops.append(point)

    # -------------------------------------------------
    # Create the two replacement edges
    # -------------------------------------------------
    edge_data_ab = data.copy()
    edge_data_ab["length"] = line_ab_metric.length
    edge_data_ab["geometry"] = line_ab

    edge_data_bc = data.copy()
    edge_data_bc["length"] = line_bc_metric.length
    edge_data_bc["geometry"] = line_bc

    graph_of_stops.add_edge(
        node1,
        point.id,
        **edge_data_ab
    )

    graph_of_stops.add_edge(
        point.id,
        node2,
        **edge_data_bc
    )

    # Remove ONLY the exact original MultiDiGraph edge
    graph_of_stops.remove_edge(
        node1,
        node2,
        edge_key
    )
        
def calculate_coordinate_along_edge(edge, position):
    # Calculate the coordinate along the edge at the given position
    point = edge.interpolate(position)
    return point.x, point.y

In [22]:
# CREATING STOPS
# It should return a list of coordinates/nodes for stop and a graph of stops
# if residential area, check if the population density

def place_stops_on_roads(transpo_points, non_transpo_points, graph_of_stops, list_of_stops):
    
    for point in transpo_points:
        add_point_to_graph(point, graph_of_stops, list_of_stops, True)
        
    for point in non_transpo_points:
        add_point_to_graph(point, graph_of_stops, list_of_stops, False)

In [23]:
def merge_stops(stops_list, max_distance_m=20):
    """
    Merge nearby existing jeepney stops while preserving
    distinct cleaned transportation-source identities.

    Rules:
    1. Transportation stops from different cleaned sources
       must never be merged.
    2. Transportation stops from the same source may be
       merged if they are within max_distance_m.
    3. Non-transportation stops may be merged with nearby
       transportation stops, preferring the nearest one.
    4. Non-transportation stops may also be merged with
       other nearby non-transportation stops.
    """

    stop_dict = {}

    if not stops_list:
        return stop_dict

    stops = sorted(
        stops_list,
        key=lambda stop: (stop.lat, stop.long),
        reverse=True
    )

    # Preserve the original grouping procedure.
    grouped_stops = []
    current_group = [stops[0]]

    for stop in stops[1:]:

        previous_stop = current_group[-1]

        distance_m = geodesic(
            (previous_stop.lat, previous_stop.long),
            (stop.lat, stop.long)
        ).meters

        if distance_m <= max_distance_m:
            current_group.append(stop)
        else:
            grouped_stops.append(current_group)
            current_group = [stop]

    grouped_stops.append(current_group)

    # Process each group independently.
    for group in grouped_stops:

        if len(group) <= 1:
            continue

        # Prioritize transportation stops so that
        # non-transportation stops can merge into them.
        ordered_group = (
            [stop for stop in group if stop.isTranspo]
            +
            [stop for stop in group if not stop.isTranspo]
        )

        representatives = []

        for stop in ordered_group:

            if (
                stop.isTranspo
                and stop.transport_source_id is None
            ):
                raise ValueError(
                    f"Transportation stop {stop.id} "
                    "has no transport_source_id."
                )

            candidates = []

            for representative in representatives:

                # Transportation stops can only merge
                # with the same cleaned transportation source.
                if stop.isTranspo:

                    if not representative.isTranspo:
                        continue

                    if (
                        stop.transport_source_id
                        != representative.transport_source_id
                    ):
                        continue

                distance_m = geodesic(
                    (stop.lat, stop.long),
                    (
                        representative.lat,
                        representative.long
                    )
                ).meters

                if distance_m <= max_distance_m:
                    candidates.append(
                        (distance_m, representative)
                    )

            if candidates:

                # Merge into the nearest compatible stop.
                _, selected_representative = min(
                    candidates,
                    key=lambda item: (
                        item[0],
                        item[1].id
                    )
                )

                stop_dict[stop.id] = (
                    selected_representative
                )

            else:

                # Preserve this stop as an independent
                # representative.
                representatives.append(stop)

    return stop_dict

In [24]:
def change_merged_stops(route_list, non_transpo_points_Manila, transpo_points_Manila, stop_dict):
    transpo_list = []
    non_transpo_list = []
    for route in route_list:
        for stop in route:
            if stop.id in stop_dict:
                index = route.index(stop)
                route[index] = stop_dict[stop.id]
    
    # Make sure there are no duplicates
    for i, route in enumerate(route_list):
        seen = set()
        unique_stops = []
        for stop in route:
            if stop.id not in seen:
                unique_stops.append(stop)
                seen.add(stop.id)
        route_list[i] = unique_stops
                
    for stop in non_transpo_points_Manila:
        if stop.id not in stop_dict:
            non_transpo_list.append(stop)
            
    for stop in transpo_points_Manila:
        if stop.id not in stop_dict:
            transpo_list.append(stop)
            
    return transpo_list, non_transpo_list

### Population and Flood Hazard Attributes

In [25]:
def build_population_reference(
    zone_graph,
    percentile=POPULATION_DENOMINATOR_PERCENTILE
):
    """
    Build the fixed zone-population lookup and the fixed
    denominator used to normalize P_n(G).

    The denominator is the specified percentile of the
    population_sum values of all final Level-2 amenity zones.
    """

    if not 0 < percentile <= 100:
        raise ValueError(
            "Population denominator percentile must be "
            "greater than 0 and less than or equal to 100."
        )

    zone_population = {}

    for zone_id, data in zone_graph.nodes(data=True):
        geometry = data.get("geometry")

        if (
            geometry is None
            or geometry.geom_type not in (
                "Polygon",
                "MultiPolygon"
            )
        ):
            continue

        if "population_sum" not in data:
            continue

        zone_population[zone_id] = float(
            data["population_sum"]
        )

    if not zone_population:
        raise ValueError(
            "No Level-2 amenity zones with population_sum "
            "were found."
        )

    population_values = np.asarray(
        list(zone_population.values()),
        dtype=float
    )

    population_denominator = float(
        np.percentile(
            population_values,
            percentile
        )
    )

    if population_denominator <= 0:
        raise ValueError(
            "Population normalization denominator "
            "must be greater than 0."
        )

    return (
        zone_population,
        population_denominator
    )

In [26]:
def assign_dynamic_population_scores(
    network_graph,
    zone_population,
    population_denominator
):
    """
    Compute network-dependent population demand.

    For each zone z:

        N_z(G) = number of active stops in network G
                 associated with zone z

    For each active stop n:

        p_n(G) = sum[
            pop(z) / N_z(G)
            for z associated with n
        ]

        P_n(G) = min(
            p_n(G) / population_denominator,
            1.0
        )

    Only stops actually present in network_graph are counted.
    """

    if population_denominator <= 0:
        raise ValueError(
            "population_denominator must be greater than 0."
        )

    # ---------------------------------------------------
    # Determine active stop-to-zone associations
    # ---------------------------------------------------
    stop_to_zones = {}
    zone_stop_counts = {}

    for stop_id, node_data in network_graph.nodes(
        data=True
    ):
        zone_ids = {
            zone_id
            for zone_id
            in (node_data.get("zone_ids") or [])
            if zone_id in zone_population
        }

        stop_to_zones[stop_id] = zone_ids

        for zone_id in zone_ids:
            zone_stop_counts[zone_id] = (
                zone_stop_counts.get(
                    zone_id,
                    0
                )
                + 1
            )

    # ---------------------------------------------------
    # Compute p_n(G) and P_n(G)
    # ---------------------------------------------------
    for stop_id, zone_ids in stop_to_zones.items():

        population_raw = 0.0

        for zone_id in zone_ids:
            active_stop_count = (
                zone_stop_counts[zone_id]
            )

            population_raw += (
                zone_population[zone_id]
                / active_stop_count
            )

        population_score = min(
            population_raw
            / population_denominator,
            1.0
        )

        network_graph.nodes[
            stop_id
        ]["population_raw"] = float(
            population_raw
        )

        network_graph.nodes[
            stop_id
        ]["population_score"] = float(
            population_score
        )

    return network_graph

In [27]:
def assign_existing_stop_population_attributes(
    list_of_stops,
    graph_of_stops,
    zone_graph,
    association_buffer_m=30
):
    """
    Associate existing jeepney stops with final Level-2 amenity zones
    and assign the intermodal indicator.

    Population demand is network-dependent. Therefore, population_raw
    and population_score remain 0.0 at this stage and are calculated
    after the final existing route network G has been constructed.

    Existing stops are associated spatially with final Level-2 amenity
    zones whose 30 m buffered geometry intersects the snapped stop point.
    """

    # ---------------------------------------------------
    # Build spatial index for buffered Level-2 zones
    # ---------------------------------------------------
    zone_index = rtree_index.Index()
    zone_records = {}
    position_to_zone_id = {}

    position = 0

    for zone_id, zone_data in zone_graph.nodes(data=True):
        geometry = zone_data.get("geometry")

        if (
            geometry is None
            or geometry.geom_type not in ["Polygon", "MultiPolygon"]
        ):
            continue

        buffered_geometry = buffer_geometry_m(
            geometry,
            association_buffer_m
        )

        zone_records[zone_id] = {
            "buffered_geometry": buffered_geometry,
            "population_sum": float(
                zone_data.get("population_sum", 0.0)
            )
        }

        zone_index.insert(
            position,
            buffered_geometry.bounds
        )

        position_to_zone_id[position] = zone_id
        position += 1

    # ---------------------------------------------------
    # Compute raw population value p_n for every stop
    # ---------------------------------------------------
    for stop in list_of_stops:
        # Use final snapped coordinates
        stop_point = Point(
            stop.long,
            stop.lat
        )

        associated_zone_ids = []

        for position in zone_index.intersection(
            stop_point.bounds
        ):
            zone_id = position_to_zone_id[position]
            zone_record = zone_records[zone_id]

            if zone_record[
                "buffered_geometry"
            ].intersects(stop_point):
                associated_zone_ids.append(
                    zone_id
                )

        # Save zone associations
        stop.zone_ids = associated_zone_ids
        
        # Population demand will be calculated after the final existing route network G has been constructed.
        stop.population_raw = 0.0
        stop.population_score = 0.0

        # T_n
        stop.intermodal_indicator = (
            1 if stop.isTranspo else 0
        )

    # ---------------------------------------------------
    # Normalize p_n -> P_n and copy attributes to graph
    # ---------------------------------------------------
    for stop in list_of_stops:
        node_data = graph_of_stops.nodes[
            stop.id
        ]

        node_data["zone_ids"] = list(
            stop.zone_ids
        )

        node_data["population_raw"] = 0.0
        node_data["population_score"] = 0.0

        node_data[
            "intermodal_indicator"
        ] = int(
            stop.intermodal_indicator
        )

    return (
        list_of_stops,
        graph_of_stops
    )

In [28]:
def load_flood_hazard_layer(filepath):
    flood_gdf = gpd.read_file(filepath)

    # Make sure the flood layer uses the same CRS as the stop coordinates
    if flood_gdf.crs is None:
        raise ValueError(
            f"Flood layer has no CRS: {filepath}"
        )

    flood_gdf = flood_gdf.to_crs("EPSG:4326").copy()

    # Convert Var to integer hazard level
    flood_gdf["Var"] = pd.to_numeric(
        flood_gdf["Var"],
        errors="raise"
    ).astype(int)

    # Validate expected Project NOAH hazard classes
    valid_values = {1, 2, 3}
    actual_values = set(flood_gdf["Var"].unique())

    if not actual_values.issubset(valid_values):
        raise ValueError(
            f"Unexpected flood hazard values in {filepath}: "
            f"{sorted(actual_values)}"
        )

    return flood_gdf

In [29]:
def get_flood_hazard_at_point(point, flood_gdf):
    """
    Returns:
        0 = no flood hazard
        1 = low
        2 = medium
        3 = high

    If the point intersects multiple hazard polygons,
    the highest hazard level is retained.
    """

    possible_matches = list(
        flood_gdf.sindex.intersection(
            point.bounds
        )
    )

    max_hazard = 0

    for pos in possible_matches:
        row = flood_gdf.iloc[pos]

        if row.geometry.intersects(point):
            max_hazard = max(
                max_hazard,
                int(row["Var"])
            )

    return max_hazard

In [30]:
def assign_flood_attributes(
    list_of_stops,
    graph_of_stops,
    flood_5_gdf,
    flood_25_gdf,
    flood_100_gdf
):
    for stop in list_of_stops:
        stop_point = Point(
            stop.long,
            stop.lat
        )

        # Individual return-period hazard levels
        stop.flood_5 = get_flood_hazard_at_point(
            stop_point,
            flood_5_gdf
        )

        stop.flood_25 = get_flood_hazard_at_point(
            stop_point,
            flood_25_gdf
        )

        stop.flood_100 = get_flood_hazard_at_point(
            stop_point,
            flood_100_gdf
        )

        # Combined normalized flood hazard score H_n
        stop.flood_score = (
            stop.flood_5
            + stop.flood_25
            + stop.flood_100
        ) / 9.0

        # Store the same attributes in the NetworkX stop node
        graph_of_stops.nodes[stop.id]["flood_5"] = (
            stop.flood_5
        )

        graph_of_stops.nodes[stop.id]["flood_25"] = (
            stop.flood_25
        )

        graph_of_stops.nodes[stop.id]["flood_100"] = (
            stop.flood_100
        )

        graph_of_stops.nodes[stop.id]["flood_score"] = (
            stop.flood_score
        )

    return list_of_stops, graph_of_stops

In [31]:
def validate_existing_stop_attributes(
    list_of_stops,
    graph_of_stops
):
    for stop in list_of_stops:
        assert (
            0.0
            <= stop.population_score
            <= 1.0
        )

        assert (
            stop.intermodal_indicator
            in (0, 1)
        )

        assert stop.flood_5 in (0, 1, 2, 3)
        assert stop.flood_25 in (0, 1, 2, 3)
        assert stop.flood_100 in (0, 1, 2, 3)

        assert (
            0.0
            <= stop.flood_score
            <= 1.0
        )

        assert stop.id in graph_of_stops

        node_data = graph_of_stops.nodes[
            stop.id
        ]

        assert math.isclose(
            node_data["population_score"],
            stop.population_score
        )

        assert (
            node_data["intermodal_indicator"]
            == stop.intermodal_indicator
        )

        assert math.isclose(
            node_data["flood_score"],
            stop.flood_score
        )

    print(
        "Existing stop attribute validation passed."
    )

In [32]:
def validate_network_population_scores(
    network_graph,
    zone_population,
    population_denominator
):
    # Count active stops associated with each zone
    stop_to_zones = {}
    zone_stop_counts = {}

    for stop_id, data in network_graph.nodes(data=True):

        zone_ids = {
            zone_id
            for zone_id in (
                data.get("zone_ids") or []
            )
            if zone_id in zone_population
        }

        stop_to_zones[stop_id] = zone_ids

        for zone_id in zone_ids:
            zone_stop_counts[zone_id] = (
                zone_stop_counts.get(
                    zone_id,
                    0
                )
                + 1
            )

    # Verify p_n(G) and P_n(G)
    for stop_id, zone_ids in stop_to_zones.items():

        expected_population_raw = sum(
            zone_population[zone_id]
            / zone_stop_counts[zone_id]
            for zone_id in zone_ids
        )

        expected_population_score = min(
            expected_population_raw
            / population_denominator,
            1.0
        )

        actual_population_raw = (
            network_graph.nodes[
                stop_id
            ].get("population_raw")
        )

        actual_population_score = (
            network_graph.nodes[
                stop_id
            ].get("population_score")
        )

        assert actual_population_raw is not None, (
            f"Node {stop_id} has no population_raw."
        )

        assert actual_population_score is not None, (
            f"Node {stop_id} has no population_score."
        )

        assert math.isclose(
            actual_population_raw,
            expected_population_raw,
            rel_tol=1e-9,
            abs_tol=1e-9
        ), (
            f"Incorrect population_raw for stop {stop_id}: "
            f"{actual_population_raw} != "
            f"{expected_population_raw}"
        )

        assert math.isclose(
            actual_population_score,
            expected_population_score,
            rel_tol=1e-9,
            abs_tol=1e-9
        ), (
            f"Incorrect population_score for stop {stop_id}: "
            f"{actual_population_score} != "
            f"{expected_population_score}"
        )

    print(
        "Dynamic network population validation passed."
    )

### Route Network Generation

In [33]:
def haversine(lat1, lon1, lat2, lon2):
    # Use geopy's geodesic function to calculate the distance
    distance = geodesic((lat1, lon1), (lat2, lon2)).kilometers
    return distance

In [34]:
def generate_route_network(routes, graph_of_stops):
    overall_graph = nx.MultiDiGraph() # The route network graph
    route_network = []
    reverse_route_network = []
    num_routes = 0 # Count number of routes
    
    # Check first if there are paths
    for route in routes:
        prev_stop = None
        for stop in route:
            if prev_stop == None:
                prev_stop = stop
                continue
            
            if nx.has_path(graph_of_stops, prev_stop.id, stop.id):
                print(f"{prev_stop.id} - {stop.id}")
                prev_stop = stop
            else:
                route_index = routes.index(route)
                stop_index = route.index(prev_stop)
                routes[route_index] = route[:stop_index+1]
                break
        print()
    
    new_routes = []
    for route in routes:
        if len(route) > 3:
            new_routes.append(route)
    routes = new_routes
    
    print()
    print("Real route")
    for route in routes:
        route_id = f'{num_routes}-A' # This will be used as a key for the edge
        route_list = []
        prev_stop = None
        totalDistance = 0
        
        for stop in route: 
            if prev_stop == None:
                prev_stop = stop
                continue

            print(f"{prev_stop.id} - {stop.id}")
            
            path = nx.shortest_path(
                graph_of_stops,
                prev_stop.id,
                stop.id,
                weight="length"
            )
            route_list.append(path)
            
            distance_travelled = 0
            # Get the total distance from point A to point B
            for i in range(len(path)-1):
                node_data = graph_of_stops.nodes[path[i]]
                next_node_data = graph_of_stops.nodes[path[i+1]]
                distance_travelled += haversine(node_data['y'], node_data['x'], next_node_data['y'], next_node_data['x'])
            
            prev_attrs = dict(
                graph_of_stops.nodes[prev_stop.id]
            )
            prev_attrs["lat"] = prev_stop.lat
            prev_attrs["lon"] = prev_stop.long
            overall_graph.add_node(
                prev_stop.id,
                **prev_attrs
            )
            stop_attrs = dict(
                graph_of_stops.nodes[stop.id]
            )
            stop_attrs["lat"] = stop.lat
            stop_attrs["lon"] = stop.long
            overall_graph.add_node(
                stop.id,
                **stop_attrs
            )
            overall_graph.add_edge(prev_stop.id, stop.id, key=route_id, road_path = path, distance = distance_travelled) # Add edge
            prev_stop = stop
            totalDistance += distance_travelled
            
            
        print()
        new_route_obj = Route(route_list, route_id, totalDistance)
        reverse_route = get_reverse_route(new_route_obj, graph_of_stops, overall_graph)
        route_network.append(new_route_obj)
        reverse_route_network.append(reverse_route)
        num_routes += 1
               
    return route_network, overall_graph, reverse_route_network
        
        

In [35]:
# Method 1
# Iterate through each node in reverse and get the shortest path

def get_reverse_route(route, graph_of_stops, overall_graph):
    paths = route.route
    reverse_paths = paths[::-1]
    reverse_route_list = []
    reverse_route_id = route.route_id[:-1] + 'B' # Reverse Route ID
    totalDistance = 0
    
    stop_list = []
    for reverse_index, connection in enumerate(reverse_paths):
        reverse_route_path = connection[::-1]
        end_node = reverse_route_path[-1]
        
        # Getting the edges of the start node only for the first connection
        if reverse_index == 0:
            start_node = reverse_route_path[0] # Start node is critical as it should be the same as the last stop previous connection (If this is not the first connection)
                    
        if not nx.has_path(graph_of_stops, start_node, end_node):
            return None
        else:
            stop_list.append([start_node, end_node])
                
        start_node = end_node # Next connection start node is the end node of the previous connection
        
    print(stop_list)
    for connection in stop_list:
        start_node = connection[0]
        end_node = connection[-1]
        
        reverse_route = nx.shortest_path(graph_of_stops, start_node, end_node, weight='length')
        reverse_route_list.append(reverse_route)
        
        distance_travelled = 0
        # Get the total distance from point A to point B
        for i in range(len(reverse_route)-1):
            node_data = graph_of_stops.nodes[reverse_route[i]]
            next_node_data = graph_of_stops.nodes[reverse_route[i+1]]
            distance_travelled += haversine(node_data['y'], node_data['x'], next_node_data['y'], next_node_data['x'])
        
        # Finally, Add the edge
        if reverse_route[0] not in overall_graph:
            overall_graph.add_node(reverse_route[0], **graph_of_stops.nodes[reverse_route[0]])
        if reverse_route[-1] not in overall_graph:
            overall_graph.add_node(reverse_route[-1], **graph_of_stops.nodes[reverse_route[-1]])
            
        overall_graph.add_edge(reverse_route[0], reverse_route[-1], key=reverse_route_id, road_path = reverse_route, distance = distance_travelled) # Add edge
        totalDistance += distance_travelled

    # print("REVERSE ROUTE DISTANCE: ", totalDistance)
    return Route(reverse_route_list, reverse_route_id, totalDistance)

### Fitness Function

In [36]:
# Fitness function

def compute_fitness_score(
    road_snapped_network_graph,
    weight_coverage,
    weight_internal_connectivity,
    weight_external_connectivity,
    weight_vulnerability,
    weight_robustness,
    total_road_network_distance,
    graph_of_stops,
    total_potential_transport_nodes,
    min_intersections_per_route,
    min_transpo_stops_per_route
):  
    internal_connectivity, internal_details = compute_internal_connectivity(
        road_snapped_network_graph,
        target_node_ratio=TARGET_NODE_RATIO,
        min_intersections_per_route=min_intersections_per_route
    )

    external_connectivity, external_details = compute_external_connectivity(
        road_snapped_network_graph,
        total_potential_transport_nodes=total_potential_transport_nodes,
        min_transpo_stops_per_route=min_transpo_stops_per_route
    )

    weighted_internal_connectivity = weight_internal_connectivity * internal_connectivity
    weighted_external_connectivity = weight_external_connectivity * external_connectivity

    network_coverage = get_network_coverage(
        road_snapped_network_graph,
        total_road_network_distance,
        graph_of_stops
    )
    weighted_network_coverage = weight_coverage * network_coverage

    vul, q_c_vul = compute_vulnerability(road_snapped_network_graph)
    vulnerability = normalize_vul_rob(vul, q_c_vul)
    weighted_vulnerability = weight_vulnerability * vulnerability

    rob, q_c_rob = compute_robustness(road_snapped_network_graph)
    robustness = normalize_vul_rob(rob, q_c_rob)
    weighted_robustness = weight_robustness * robustness

    data = {
        "Coverage": weighted_network_coverage,
        "Internal Connectivity": weighted_internal_connectivity,
        "External Connectivity": weighted_external_connectivity,
        "Vulnerability": weighted_vulnerability,
        "Robustness": weighted_robustness,

        "Coverage Raw": network_coverage,
        "Internal Connectivity Raw": internal_connectivity,
        "External Connectivity Raw": external_connectivity,
        "Vulnerability Raw": vulnerability,
        "Robustness Raw": robustness,

        "Total Network Nodes": len(road_snapped_network_graph.nodes()),
        "Total Potential Transport Nodes": total_potential_transport_nodes,

        "n_i": internal_details["n_i"],
        "N_i": internal_details["N_i"],
        "r_i": internal_details["r_i"],
        "R_i": internal_details["R_i"],
        "Min Intersections Per Route": internal_details["min_intersections_per_route"],

        "n_e": external_details["n_e"],
        "N_e": external_details["N_e"],
        "r_e": external_details["r_e"],
        "R_e": external_details["R_e"],
        "Min Transpo Stops Per Route": external_details["min_transpo_stops_per_route"],

        "Target Node Ratio": TARGET_NODE_RATIO,

        "q_c Vulnerability": q_c_vul,
        "q_c Robustness": q_c_rob,
    }

    total_weight = (
        weight_coverage
        + weight_internal_connectivity
        + weight_external_connectivity
        + weight_vulnerability
        + weight_robustness
    )

    weighted_sum = (
        weighted_network_coverage
        + weighted_internal_connectivity
        + weighted_external_connectivity
        + weighted_vulnerability
        + weighted_robustness
    )

    fitness_score = 0.0 if total_weight <= 0 else weighted_sum / total_weight

    return fitness_score, data

def get_total_unique_route_families(network):
    return len({
        key.split('-')[0]
        for _, _, key in network.edges(keys=True)
        if isinstance(key, str) and '-' in key
    })

def get_route_family_stop_map(network):
    """
    Returns:
        dict[route_family] = set of stop nodes touched by that route family
    Counts A/B as one route family.
    """
    route_family_stops = {}

    for u, v, key in network.edges(keys=True):
        if not (isinstance(key, str) and '-' in key):
            continue

        route_family = key.split('-')[0]
        if route_family not in route_family_stops:
            route_family_stops[route_family] = set()

        route_family_stops[route_family].add(u)
        route_family_stops[route_family].add(v)

    return route_family_stops

def compute_internal_connectivity(
    network,
    target_node_ratio=TARGET_NODE_RATIO,
    min_intersections_per_route=2
):
    total_nodes = len(network.nodes())
    if total_nodes == 0:
        return 0.0, {
            "n_i": 0,
            "N_i": 0,
            "r_i": 0,
            "R_i": 0,
            "min_intersections_per_route": min_intersections_per_route,
        }

    route_family_stops = get_route_family_stop_map(network)
    total_route_families = len(route_family_stops)

    if total_route_families == 0:
        return 0.0, {
            "n_i": 0,
            "N_i": max(1, int(total_nodes * target_node_ratio)),
            "r_i": 0,
            "R_i": 0,
            "min_intersections_per_route": min_intersections_per_route,
        }

    # Identify intersection nodes: nodes touched by >1 unique route family
    node_route_families = {}
    for route_family, stops in route_family_stops.items():
        for stop in stops:
            node_route_families.setdefault(stop, set()).add(route_family)

    intersection_nodes = {
        node for node, families in node_route_families.items()
        if len(families) > 1
    }

    n_i = len(intersection_nodes)
    N_i = max(1, int(total_nodes * target_node_ratio))

    # r_i = routes with at least n intersection nodes
    r_i = 0
    for route_family, stops in route_family_stops.items():
        intersection_count = sum(1 for stop in stops if stop in intersection_nodes)
        if intersection_count >= min_intersections_per_route:
            r_i += 1

    R_i = total_route_families

    internal_connectivity = 0.5 * (
        min(n_i / N_i, 1.0) +
        (r_i / R_i if R_i > 0 else 0.0)
    )

    details = {
        "total_nodes": total_nodes,
        "n_i": n_i,
        "N_i": N_i,
        "r_i": r_i,
        "R_i": R_i,
        "min_intersections_per_route": min_intersections_per_route,
    }

    return internal_connectivity, details

def compute_external_connectivity(
    network,
    total_potential_transport_nodes,
    min_transpo_stops_per_route=2
):
    total_nodes = len(network.nodes())
    if total_nodes == 0:
        return 0.0, {
            "total_nodes": 0,
            "n_e": 0,
            "N_e": max(1, total_potential_transport_nodes),
            "r_e": 0,
            "R_e": 0,
            "min_transpo_stops_per_route": min_transpo_stops_per_route,
        }

    route_family_stops = get_route_family_stop_map(network)
    total_route_families = len(route_family_stops)

    transpo_nodes = {
        node for node, node_data in network.nodes(data=True)
        if node_data.get("isTranspo", False) is True
    }

    n_e = len(transpo_nodes)
    N_e = max(1, total_potential_transport_nodes)

    if total_route_families == 0:
        return 0.0, {
            "total_nodes": total_nodes,
            "n_e": n_e,
            "N_e": N_e,
            "r_e": 0,
            "R_e": 0,
            "min_transpo_stops_per_route": min_transpo_stops_per_route,
        }

    r_e = 0
    for route_family, stops in route_family_stops.items():
        transpo_count = sum(1 for stop in stops if stop in transpo_nodes)
        if transpo_count >= min_transpo_stops_per_route:
            r_e += 1

    R_e = total_route_families

    external_connectivity = 0.5 * (
        min(n_e / N_e, 1.0) +
        (r_e / R_e if R_e > 0 else 0.0)
    )

    details = {
        "total_nodes": total_nodes,
        "n_e": n_e,
        "N_e": N_e,
        "r_e": r_e,
        "R_e": R_e,
        "min_transpo_stops_per_route": min_transpo_stops_per_route,
    }

    return external_connectivity, details
    
def largest_weakly_connected_component_size(G):
    if len(G) == 0:
        return 0
    return len(max(nx.weakly_connected_components(G), key=len))

def compute_vulnerability(network_graph):
    G = network_graph.copy()
    N = len(G.nodes())
    if N == 0:
        return 0.0, 0.0

    num_removals = max(1, int(N * 0.1))
    s_values = []

    for k in range(1, num_removals + 1):
        if len(G.nodes()) == 0:
            break

        deg_view = G.degree()
        max_node, _ = max_degree_node_first(deg_view)
        G.remove_node(max_node)

        lwcc_size = largest_weakly_connected_component_size(G)
        s_q = lwcc_size / N
        s_values.append(s_q)

    q_c = num_removals / N
    R_prime = sum(s_values) / N
    return R_prime, q_c

def compute_robustness(network_graph):
    G = network_graph.copy()
    N = len(G.nodes())
    if N == 0:
        return 0.0, 0.0

    num_removals = max(1, int(N * 0.1))
    s_values = []

    for k in range(1, num_removals + 1):
        if len(G.nodes()) == 0:
            break

        selected_node = random.choice(list(G.nodes()))
        G.remove_node(selected_node)

        lwcc_size = largest_weakly_connected_component_size(G)
        s_q = lwcc_size / N
        s_values.append(s_q)

    q_c = num_removals / N
    R_prime = sum(s_values) / N
    return R_prime, q_c

def normalize_vul_rob(vul_rob, q_c):
    return 0.0 if q_c <= 0 else vul_rob / q_c

def max_degree_node_first(deg_view):
    max_d = -1
    max_n = None
    for n, d in deg_view:
        if d > max_d:
            max_d = d
            max_n = n
    return max_n, max_d

def get_node_with_degree(node_degrees, degree):
    # Iterate over the DegreeView object to find the node with the specified degree
    for node, _ in node_degrees:
        if _ == degree:
            return node
    return None  # Return None if no node with the specified degree is found
    
def get_network_coverage(network_graph, total_road_network_distance, graph_of_stops):
    
    travelled_distance = 0
    edge_set = [] # This will contain all the edges travelled
    path_set = []
    
    for node1, node2, edge_data in network_graph.edges(data=True):
        string_key1 = f"{node1}_{node2}"
        string_key2 = f"{node2}_{node1}"
        
        if string_key1 not in edge_set and string_key2 not in edge_set:
            edge_set.append(string_key1)
            edge_set.append(string_key2)
            path_travelled = edge_data['road_path']
            
            prev_stop = None
            for stop in path_travelled:
                if prev_stop == None:
                    prev_stop = stop
                    continue
                    
                key = f"{prev_stop}_{stop}"
                edge = graph_of_stops.get_edge_data(prev_stop, stop)
                data = list(edge.values())[0]
                edge_highway = data['highway']
                
                rows_with_lists['highway'].apply(check_list)
                if key not in path_set:
                    if isinstance(edge_highway, list) and (any(element in edge_highway for element in filter_options)) or edge_highway in filter_options:
                        travelled_distance += data['length']
                        path_set.append(key)
                        
                prev_stop = stop
            
    return travelled_distance / total_road_network_distance

# Get the roads whose highway types are part of the list
def check_list(lst):
    return any(x in filter_options for x in lst)

### Network Analysis Metrics

In [37]:
# Longest Route
def get_longest_route(network_routes):
    sorted_routes = sorted(network_routes, key=lambda x: x.length, reverse=True)
    longest_route = sorted_routes[0]
    
    stop_list = [] # Get all the stops
    for connection in longest_route.route:
        if connection[0] not in stop_list:
            stop_list.append(connection[0])
        if connection[-1] not in stop_list:
            stop_list.append(connection[-1])
    
    return longest_route, stop_list

# Shortest Route
def get_shortest_route(network_routes):
    sorted_routes = sorted(network_routes, key=lambda x: x.length, reverse=True)
    shortest_route = sorted_routes[-1]
    
    stop_list = [] # Get all the stops
    for connection in shortest_route.route:
        if connection[0] not in stop_list:
            stop_list.append(connection[0])
        if connection[-1] not in stop_list:
            stop_list.append(connection[-1])
    
    return shortest_route, stop_list

# Average Route Length
def get_average_route_length(network_routes):
    sum_length = sum([n.length for n in network_routes])
    average_length = sum_length / len(network_routes)
    return average_length

def get_longest_path_lengths(snapped_road_network_graph):
    path_lengths = []
    for node in snapped_road_network_graph.nodes():
        lengths = nx.single_source_dijkstra_path_length(snapped_road_network_graph, node, weight='distance')
        list_lengths = [n for target_node, n in lengths.items()]
        path_lengths.append(max(list_lengths))
    return path_lengths

#Network Diameter
def get_network_diameter(network_graph):
    path_lengths = get_longest_path_lengths(network_graph) # Get the sum of all possible
    max_path_length = max(path_lengths)
    return max_path_length

def average_degree(graph):
    degrees = [degree for _, degree in graph.degree()]
    avg_degree = sum(degrees) / len(degrees)
    return avg_degree

def average_number_of_stops_per_route(network_routes):
    num_stops = []
    for route in network_routes:
        seen = set()
        for connection in route.route:
            if connection[0] not in seen:
                seen.add(connection[0])
            
            if connection[-1] not in seen:
                seen.add(connection[-1])
                
        num_stops.append(len(seen))
            
    avg_stops = sum(num_stops) / len(num_stops)
    return avg_stops

In [38]:
# Node Centrality
def get_node_centrality(network_graph):
    degree_centrality = nx.degree_centrality(network_graph)
    betweenness_centrality = nx.betweenness_centrality(network_graph)
    closeness_centrality = nx.closeness_centrality(network_graph)
    
    return degree_centrality, betweenness_centrality, closeness_centrality

### Visualizations

In [39]:
def add_markers(routes, map):
    added = set()
    for route in routes:
        for stop in route:
            if stop.id not in added:
                #popup_text = f"Name: {stop.name}<br>Type: {stop.a_type}<br>Coordinates: {stop.getLat()}, {stop.getLong()}"
                lat = stop.lat
                long = stop.long
                
                if stop.isTranspo:
                    folium.Marker(location=[lat, long],icon=folium.Icon(color='green')).add_to(map)
                else:
                    folium.Marker(location=[lat, long],icon=folium.Icon(color='blue')).add_to(map)
                added.add(stop.id)

In [40]:
# This is to visualize the stops
def plot_stops_on_map(stops):
    # Iterate over the nodes in the network
    map_center = (14.599512, 120.984222)
    network_map = folium.Map(location=map_center, zoom_start=10, tiles='openstreetmap')
    for stop in stops:
        
        if stop.isTranspo:
            marker_color = 'blue'
        else:
            marker_color = 'red'
            
        folium.Marker(location=[stop.lat, stop.long], popup=f"Transportation: {stop.isTranspo}", icon=folium.Icon(color=marker_color)).add_to(network_map)
        
    return network_map

In [41]:
def improved_route_plot(graph, route, color, map, route_id):
    for connection in route.route:
        route_coords = [(graph.nodes[node]['y'], graph.nodes[node]['x']) for node in connection]
        
        # If this is the first connection, mark the first stop
        if route.route.index(connection) == 0:
            lat = graph.nodes[connection[0]]['y']
            long = graph.nodes[connection[0]]['x']
            isTranspo = graph.nodes[connection[0]]['isTranspo']
            
            folium.CircleMarker(
            location=[lat, long],
            radius=6,  # Adjust the radius as needed
            popup=f"Transportation: {isTranspo}\n{lat},{long}",
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7
            ).add_to(map)
            
        lat = graph.nodes[connection[-1]]['y']
        long = graph.nodes[connection[-1]]['x']
        isTranspo = graph.nodes[connection[-1]]['isTranspo']
        
        folium.CircleMarker(
        location=[lat, long],
        radius=8,  # The size of the circles
        popup=f"Transportation: {isTranspo}\n{lat},{long}",
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7
        ).add_to(map)
        
        folium.PolyLine(route_coords, popup=f"{route_id}", color=color, opacity=0.8).add_to(map)
    
    return map

In [42]:
def get_network_bounds(network_obj, graph_of_stops, pad_ratio=0.03):
    xs = []
    ys = []

    for route in network_obj.routes:
        for connection in route.route:
            for node in connection:
                xs.append(graph_of_stops.nodes[node]["x"])  # lon
                ys.append(graph_of_stops.nodes[node]["y"])  # lat

    if not xs or not ys:
        return None

    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    dx = (max_x - min_x) * pad_ratio
    dy = (max_y - min_y) * pad_ratio

    return [[min_y - dy, min_x - dx], [max_y + dy, max_x + dx]]

In [43]:
def save_route_map_html(
    network_obj,
    graph_of_stops,
    output_html_path,
    map_center=(14.599512, 120.984222),
    zoom_start=10,
    tiles="OpenStreetMap",
    zoom_control=False,
    pad_ratio=0.03,
    color_seed=42
):
    """
    Creates a Folium route map, fits bounds to the whole network,
    and saves it as HTML.
    """
    random.seed(color_seed)

    colors = [
        "Red", "Green", "Blue", "Orange", "Purple", "Magenta", "Maroon",
        "Olive", "Navy", "Fuchsia", "Coral", "Indigo", "Violet", "Brown",
        "DarkGreen", "DarkBlue", "DarkRed", "DarkCyan", "DarkMagenta",
        "DarkOrange", "DarkViolet", "DarkTurquoise", "DarkSlateGray",
        "SaddleBrown", "SeaGreen", "MidnightBlue", "ForestGreen", "Sienna",
        "SteelBlue", "RoyalBlue", "SlateBlue", "MediumSeaGreen",
        "MediumSlateBlue", "MediumPurple", "MediumTurquoise", "Crimson",
        "Goldenrod", "Chocolate", "FireBrick", "Peru", "RosyBrown",
        "IndianRed", "SlateGray", "DodgerBlue", "LightSeaGreen",
        "MediumAquamarine", "CadetBlue"
    ]

    m = folium.Map(
        location=map_center,
        zoom_start=zoom_start,
        tiles=tiles,
        zoom_control=zoom_control
    )
    """
    folium.TileLayer(
        tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        attr="© OpenStreetMap contributors",
        name="OpenStreetMap",
        overlay=False,
        control=True
    ).add_to(m)
    """

    for route in network_obj.routes:
        chosen_color = random.choice(colors)
        improved_route_plot(graph_of_stops, route, chosen_color, m, route.route_id)

    bounds = get_network_bounds(network_obj, graph_of_stops, pad_ratio=pad_ratio)
    if bounds is not None:
        m.fit_bounds(bounds)

    os.makedirs(os.path.dirname(output_html_path), exist_ok=True)
    m.save(output_html_path)
    print(f"Saved HTML map to: {output_html_path}")

## Main

### Read Data

In [44]:
# File Paths
pikl_filepath = "Original Jeepney Routes/Saved Networks/"
map_filepath = "Original Jeepney Routes/Maps/"

# Load Flood Hazard Layers
flood_5_gdf = load_flood_hazard_layer(
    "Flood Data/MetroManila_Flood_5year.shp"
)
flood_25_gdf = load_flood_hazard_layer(
    "Flood Data/MetroManila_Flood_25year.shp"
)
flood_100_gdf = load_flood_hazard_layer(
    "Flood Data/MetroManila_Flood_100year.shp"
)

# Load Zone Networks
manila_zone_graph = import_networks(
    "Saved Networks/Manila/Manila_Zone_Network.pkl"
)
makati_zone_graph = import_networks(
    "Saved Networks/Makati/Makati_Zone_Network.pkl"
)
mandaluyong_zone_graph = import_networks(
    "Saved Networks/Mandaluyong/Mandaluyong_Zone_Network.pkl"
)

In [45]:
(ZONE_POPULATION_MANILA, POPULATION_DENOMINATOR_MANILA) = build_population_reference(
    manila_zone_graph,
    percentile=POPULATION_DENOMINATOR_PERCENTILE
)

In [46]:
import ast
import json
from pathlib import Path

GA_NOTEBOOK_FILE = Path(
    "Manila_RouteConnection_GA_ModifiedImprovement.ipynb"
)

def load_ga_function_definitions(function_names):
    """
    Load only explicitly named top-level functions
    from the trusted local GA notebook.

    Does not execute the notebook's other cells.
    """

    if not GA_NOTEBOOK_FILE.exists():
        raise FileNotFoundError(
            f"GA notebook not found: {GA_NOTEBOOK_FILE}"
        )

    with open(GA_NOTEBOOK_FILE, encoding="utf-8") as f:
        notebook_data = json.load(f)

    requested = set(function_names)
    selected = {}

    for cell in notebook_data["cells"]:

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")

        if isinstance(source, list):
            source = "".join(source)

        try:
            tree = ast.parse(source)
        except SyntaxError as exc:
            # Do not silently skip a requested function.
            if any(
                f"def {name}(" in source
                for name in requested
            ):
                raise SyntaxError(
                    "Could not parse a GA notebook cell "
                    "containing a required function."
                ) from exc
            continue

        for node in tree.body:
            if (
                isinstance(node, ast.FunctionDef)
                and node.name in requested
            ):
                selected[node.name] = node

    missing = requested - set(selected)

    if missing:
        raise ValueError(
            f"GA functions not found: {sorted(missing)}"
        )

    function_module = ast.Module(
        body=[
            selected[name]
            for name in function_names
        ],
        type_ignores=[]
    )

    ast.fix_missing_locations(function_module)

    exec(
        compile(
            function_module,
            str(GA_NOTEBOOK_FILE),
            "exec"
        ),
        globals()
    )

    print(
        "Loaded GA functions:",
        ", ".join(function_names)
    )

In [47]:
TRANSPORT_POI_DEDUP_DISTANCE_M = 20.0

load_ga_function_definitions([
    "normalize_transport_name",
    "add_points_to_graph"
])

# Original Manila transportation POIs used by the GA.
ga_initial_amenity_network = import_networks(
    "Saved Networks/Manila/"
    "Manila_Initial_Network.pkl"
)

cleaned_source_graph = nx.Graph()

add_points_to_graph(
    ga_initial_amenity_network,
    cleaned_source_graph,
    dedup_distance_m=TRANSPORT_POI_DEDUP_DISTANCE_M
)

Loaded GA functions: normalize_transport_name, add_points_to_graph
Transportation POI cleaning:
raw_transport_pois: 507
cleaned_transport_points: 419
removed_duplicate_pois: 88
duplicate_groups: 82
largest_raw_group: 6


{'raw_transport_pois': 507,
 'cleaned_transport_points': 419,
 'removed_duplicate_pois': 88,
 'duplicate_groups': 82,
 'largest_raw_group': 6}

In [48]:
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

# =========================================================
# MATCH GEOJSON TRANSPORTATION POIs TO GA SOURCE POINTS
# =========================================================

MATCH_TOLERANCE_M = 2.0

# 1. Collect original GeoJSON transportation points.
geo_transport_gdf = manila_transpo_gdf.to_crs(
    "EPSG:32651"
)

if not geo_transport_gdf.geometry.geom_type.eq("Point").all():
    raise ValueError(
        "GeoJSON transportation features must be Points."
    )

geo_ids = list(geo_transport_gdf.index)

geo_coordinates = np.array([
    [geometry.x, geometry.y]
    for geometry in geo_transport_gdf.geometry
])

geo_names = [
    normalize_transport_name(value)
    for value in geo_transport_gdf["name"]
]


# 2. Collect the corresponding raw GA transportation points.
ga_transport_points = {
    node_id: data
    for node_id, data in (
        ga_initial_amenity_network.nodes(data=True)
    )
    if (
        data.get("amenity") == "transportation"
        and data.get("geometry") is not None
        and data["geometry"].geom_type == "Point"
    )
}

ga_ids = list(ga_transport_points.keys())

ga_coordinates = np.array([
    [
        to_metric_geometry(
            ga_transport_points[node_id]["geometry"]
        ).x,
        to_metric_geometry(
            ga_transport_points[node_id]["geometry"]
        ).y
    ]
    for node_id in ga_ids
])

ga_names = [
    normalize_transport_name(
        ga_transport_points[node_id].get("name")
    )
    for node_id in ga_ids
]

print("GeoJSON transportation points:", len(geo_ids))
print("GA raw transportation points:", len(ga_ids))

if len(geo_ids) != len(ga_ids):
    raise ValueError(
        "The datasets have different transportation "
        "point counts. Inspect their provenance before "
        "attempting a one-to-one match."
    )


# 3. Calculate spatial distances in meters.
distance_matrix = cdist(
    geo_coordinates,
    ga_coordinates
)

# Prefer identical names when multiple POIs
# occupy the same or very similar coordinates.
geo_name_array = np.asarray(geo_names)
ga_name_array = np.asarray(ga_names)

name_conflicts = (
    (geo_name_array[:, None] != "")
    & (ga_name_array[None, :] != "")
    & (
        geo_name_array[:, None]
        != ga_name_array[None, :]
    )
)

assignment_cost = (
    distance_matrix
    + name_conflicts.astype(float) * 1000.0
)

# Find a unique matching GA point for each GeoJSON point.
geo_positions, ga_positions = linear_sum_assignment(
    assignment_cost
)


# 4. Build raw-GA-ID -> cleaned-source-ID mapping.
raw_to_cleaned_source = {}

for cleaned_id, data in (
    cleaned_source_graph.nodes(data=True)
):
    for raw_id in data.get(
        "raw_transport_source_ids", []
    ):
        raw_to_cleaned_source[raw_id] = cleaned_id


# 5. Validate every matched pair.
match_records = []

for geo_pos, ga_pos in zip(
    geo_positions,
    ga_positions
):

    geo_id = geo_ids[geo_pos]
    ga_id = ga_ids[ga_pos]

    distance_m = float(
        distance_matrix[geo_pos, ga_pos]
    )

    if ga_id not in raw_to_cleaned_source:
        raise ValueError(
            f"GA source {ga_id} has no cleaned source ID."
        )

    match_records.append({
        "GeoJSON Index": geo_id,
        "GA Raw ID": ga_id,
        "Cleaned Source ID":
            raw_to_cleaned_source[ga_id],
        "Distance (m)": distance_m,
        "GeoJSON Name": geo_names[geo_pos],
        "GA Name": ga_names[ga_pos],
        "Name Conflict":
            bool(name_conflicts[geo_pos, ga_pos])
    })

matches_df = pd.DataFrame(match_records)

print("\n=== MATCHING VALIDATION ===")

print("Total matches:", len(matches_df))

print(
    "Maximum matching distance (m):",
    matches_df["Distance (m)"].max()
)

print(
    "Median matching distance (m):",
    matches_df["Distance (m)"].median()
)

print(
    "Matches beyond tolerance:",
    int(
        (
            matches_df["Distance (m)"]
            > MATCH_TOLERANCE_M
        ).sum()
    )
)

print(
    "Named source conflicts:",
    int(matches_df["Name Conflict"].sum())
)

print(
    "Unique cleaned sources:",
    matches_df["Cleaned Source ID"].nunique()
)

display(
    matches_df.sort_values(
        "Distance (m)",
        ascending=False
    ).head(10)
)


# 6. Stop if the datasets do not match reliably.
if (
    (matches_df["Distance (m)"] > MATCH_TOLERANCE_M).any()
    or matches_df["Name Conflict"].any()
):
    raise ValueError(
        "Transportation-source matching failed validation. "
        "Inspect the displayed matches before proceeding."
    )


# 7. Create the mapping expected by check_transpo_stops().
MANILA_CANONICAL_TRANSPORT_IDS = dict(
    zip(
        matches_df["GeoJSON Index"],
        matches_df["Cleaned Source ID"]
    )
)

assert len(MANILA_CANONICAL_TRANSPORT_IDS) == len(
    manila_transpo_gdf
)

assert len(set(
    MANILA_CANONICAL_TRANSPORT_IDS.values()
)) == cleaned_source_graph.number_of_nodes()

print(
    "\nTransportation-source alignment passed."
)

print(
    "GeoJSON transportation points mapped:",
    len(MANILA_CANONICAL_TRANSPORT_IDS)
)

GeoJSON transportation points: 507
GA raw transportation points: 507

=== MATCHING VALIDATION ===
Total matches: 507
Maximum matching distance (m): 0.0
Median matching distance (m): 0.0
Matches beyond tolerance: 0
Named source conflicts: 0
Unique cleaned sources: 419


,GeoJSON Index,GA Raw ID,Cleaned Source ID,Distance (m),GeoJSON Name,GA Name,Name Conflict
0,1191,point_1191,point_1191,0.0,escolta,escolta,False
1,1192,point_1192,point_1192,0.0,stop n' shop turning point,stop n' shop turning point,False
2,1193,point_1193,point_1193,0.0,pandacan,pandacan,False
3,1194,point_1194,point_1194,0.0,santa mesa,santa mesa,False
4,1195,point_1195,point_1195,0.0,blumentritt,blumentritt,False
5,1196,point_1196,point_1196,0.0,laon laan,laon laan,False
6,1197,point_1197,point_1197,0.0,españa,españa,False
7,1198,point_1198,point_1198,0.0,vito cruz,vito cruz,False
8,1199,point_1199,point_1199,0.0,tutuban,tutuban,False
9,1200,point_1200,point_1200,0.0,san andres,san andres,False



Transportation-source alignment passed.
GeoJSON transportation points mapped: 507


### Creating the Stops

#### Manila Stops

In [50]:
# MANILA
# Remove all points that are out of bounds
tolerance = 0.0009
manila_route_stops_new = remove_if_out_of_bounds(MANILA_GRAPH, manila_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in manila_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
manila_route_stops_new = temp_array

In [149]:
# Getting all stops that are transpos (Stops that are near a transpo point are considered transpo stops)
max_distance_to_transpo_Manila = 30
# Get transpo stops
(
    transpo_points_Manila,
    non_transpo_points_Manila,
    complete_route_list_Manila
) = check_transpo_stops(
    manila_route_stops_new,
    manila_points_gdf,
    manila_idx,
    max_distance_to_transpo_Manila,
    canonical_transport_ids=(
        MANILA_CANONICAL_TRANSPORT_IDS
    )
)

In [150]:
# Merge stops that are near each other
merged_list = transpo_points_Manila + non_transpo_points_Manila
stop_dict = merge_stops(merged_list)
# Check whether distinct transportation sources
# have been merged into one stop.

stops_by_id = {
    stop.id: stop
    for stop in merged_list
}

transport_merge_conflicts = []

for discarded_id, retained_stop in stop_dict.items():

    discarded_stop = stops_by_id[discarded_id]

    if (
        discarded_stop.isTranspo
        and retained_stop.isTranspo
        and discarded_stop.transport_source_id
            != retained_stop.transport_source_id
    ):
        transport_merge_conflicts.append({
            "discarded_stop": discarded_id,
            "retained_stop": retained_stop.id,
            "discarded_source":
                discarded_stop.transport_source_id,
            "retained_source":
                retained_stop.transport_source_id
        })

print(
    "Distinct transportation-source merge conflicts:",
    len(transport_merge_conflicts)
)

if transport_merge_conflicts:
    display(
        pd.DataFrame(
            transport_merge_conflicts
        ).head(20)
    )

route_list_Manila = copy.deepcopy(complete_route_list_Manila)
transpo_points_Manila, non_transpo_points_Manila = change_merged_stops(route_list_Manila, non_transpo_points_Manila, transpo_points_Manila, stop_dict)

Distinct transportation-source merge conflicts: 0


In [151]:
# =========================================================
# VALIDATE ALL MERGED STOP DISTANCES
# =========================================================

original_stops_by_id = {
    stop.id: stop
    for stop in merged_list
}

invalid_merge_distances = []

for discarded_id, retained_stop in stop_dict.items():

    discarded_stop = original_stops_by_id[
        discarded_id
    ]

    distance_m = geodesic(
        (discarded_stop.lat, discarded_stop.long),
        (retained_stop.lat, retained_stop.long)
    ).meters

    if distance_m > 20.0 + 1e-6:
        invalid_merge_distances.append(
            (discarded_id, retained_stop.id, distance_m)
        )

print(
    "Merged stops beyond 20 meters:",
    len(invalid_merge_distances)
)

assert not invalid_merge_distances

print("Stop-merging distance validation passed.")

Merged stops beyond 20 meters: 0
Stop-merging distance validation passed.


In [153]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Manila)}")
print(f"Number of transpo stops: {len(transpo_points_Manila)}")
print(f"Ratio of transpo: {len(transpo_points_Manila) / (len(transpo_points_Manila) + len(non_transpo_points_Manila))}")
#Number of non-transpo stops: 363
#Number of transpo stops: 55
#Ratio of transpo: 0.13157894736842105

Number of non-transpo stops: 363
Number of transpo stops: 57
Ratio of transpo: 0.1357142857142857


In [154]:
# MANILA STOPCANDIDATES (Turn stops into stopCandidates and plot them on the map)
graph_of_stops_Manila = copy.deepcopy(MANILA_GRAPH)
list_of_stops_Manila = []

place_stops_on_roads(transpo_points_Manila, non_transpo_points_Manila, graph_of_stops_Manila, list_of_stops_Manila)

# ASSIGN POPULATION AND INTERMODAL ATTRIBUTES
list_of_stops_Manila, graph_of_stops_Manila = (
    assign_existing_stop_population_attributes(
        list_of_stops_Manila,
        graph_of_stops_Manila,
        manila_zone_graph
    )
)

# ASSIGN FLOOD ATTRIBUTES
assign_flood_attributes(
    list_of_stops_Manila,
    graph_of_stops_Manila,
    flood_5_gdf,
    flood_25_gdf,
    flood_100_gdf
)

validate_existing_stop_attributes(
    list_of_stops_Manila,
    graph_of_stops_Manila
)

# Visualize the stops
map = plot_stops_on_map(list_of_stops_Manila)
map.save("stop_map_manila.html")

Existing stop attribute validation passed.


In [155]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Manila.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Manila, f)
    
file_path = f'{pikl_filepath}stop_graph_Manila.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Manila, f)

#### Makati

In [64]:
# MAKATI
# Remove all points that are out of bounds
tolerance = 0.9
makati_route_stops_new = remove_if_out_of_bounds(MAKATI_GRAPH, makati_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in makati_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
makati_route_stops_new = temp_array

In [65]:
# Get transpo stops
max_distance_to_transpo_Makati = 50
transpo_points_Makati, non_transpo_points_Makati, complete_route_list_Makati = check_transpo_stops(makati_route_stops_new, makati_points_gdf, makati_idx, max_distance_to_transpo_Makati)

In [ ]:
# Merge stops in Makati that are near each other

merged_list = transpo_points_Makati + non_transpo_points_Makati
stop_dict = merge_stops(merged_list)

route_list_Makati = copy.deepcopy(complete_route_list_Makati)
transpo_points_Makati, non_transpo_points_Makati = change_merged_stops(route_list_Makati, non_transpo_points_Makati, transpo_points_Makati, stop_dict)

In [67]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Makati)}")
print(f"Number of transpo stops: {len(transpo_points_Makati)}")
print(f"Ratio of transpo: {len(transpo_points_Makati) / (len(transpo_points_Makati) + len(non_transpo_points_Makati))}")

Number of non-transpo stops: 101
Number of transpo stops: 38
Ratio of transpo: 0.2733812949640288


In [ ]:
# MAKATI STOPCANDIDATES
graph_of_stops_Makati = copy.deepcopy(MAKATI_GRAPH)
list_of_stops_Makati = []

place_stops_on_roads(transpo_points_Makati, non_transpo_points_Makati, graph_of_stops_Makati, list_of_stops_Makati)

# ASSIGN POPULATION AND INTERMODAL ATTRIBUTES
list_of_stops_Makati, graph_of_stops_Makati = (
    assign_existing_stop_population_attributes(
        list_of_stops_Makati,
        graph_of_stops_Makati,
        makati_zone_graph
    )
)

# ASSIGN FLOOD ATTRIBUTES
assign_flood_attributes(
    list_of_stops_Makati,
    graph_of_stops_Makati,
    flood_5_gdf,
    flood_25_gdf,
    flood_100_gdf
)

validate_existing_stop_attributes(
    list_of_stops_Makati,
    graph_of_stops_Makati
)

# Visualize the stops
map = plot_stops_on_map(list_of_stops_Makati)
map.save("stop_map_makati.html")

In [69]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Makati.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Makati, f)
    
file_path = f'{pikl_filepath}stop_graph_Makati.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Makati, f)

#### Mandaluyong

In [70]:
# MANDALUYONG
# Remove all points that are out of bounds
tolerance = 0.09
mandaluyong_route_stops_new = remove_if_out_of_bounds(MANDALUYONG_GRAPH, mandaluyong_route_stops, tolerance)

# Remove all routes that are less than 3 stops
temp_array = [] # For storing routes that are above 3 stops
for route in mandaluyong_route_stops_new:
    if len(route) > 3:
        temp_array.append(route)
mandaluyong_route_stops_new = temp_array

In [71]:
# Get transpo stops
max_distance_to_transpo_Mandaluyong = 50
transpo_points_Mandaluyong, non_transpo_points_Mandaluyong, complete_route_list_Mandaluyong = check_transpo_stops(mandaluyong_route_stops_new, mandaluyong_points_gdf, mandaluyong_idx, max_distance_to_transpo_Mandaluyong)

In [ ]:
merged_list = transpo_points_Mandaluyong + non_transpo_points_Mandaluyong
stop_dict = merge_stops(merged_list)

route_list_Mandaluyong = copy.deepcopy(complete_route_list_Mandaluyong)
transpo_points_Mandaluyong, non_transpo_points_Mandaluyong = change_merged_stops(route_list_Mandaluyong, non_transpo_points_Mandaluyong, transpo_points_Mandaluyong, stop_dict)

In [73]:
print(f"Number of non-transpo stops: {len(non_transpo_points_Mandaluyong)}")
print(f"Number of transpo stops: {len(transpo_points_Mandaluyong)}")
print(f"Ratio of transpo: {len(transpo_points_Mandaluyong) / (len(transpo_points_Mandaluyong) + len(non_transpo_points_Mandaluyong))}")

Number of non-transpo stops: 60
Number of transpo stops: 10
Ratio of transpo: 0.14285714285714285


In [74]:
# Test Mandaluyong Map

# Create a map centered at the initial location
map_center = (14.599512, 120.984222) # TEMPORARY WILL ZOOM TO MANILA
m = folium.Map(location=map_center, zoom_start=10, tiles='openstreetmap')

count  = 0
for i, data in mandaluyong_points_gdf.iterrows():
    if data['amenity'] == 'transportation':
        folium.Marker(location=[data['y'], data['x']], popup=f"{data['name']}", icon=folium.Icon(color='blue')).add_to(m)
        count += 1
print(count)
        
m.save('transpo_mandaluyong.html')

48


In [ ]:
# MANDALUYONG STOPCANDIDATES
graph_of_stops_Mandaluyong = copy.deepcopy(MANDALUYONG_GRAPH)
list_of_stops_Mandaluyong = []

place_stops_on_roads(transpo_points_Mandaluyong, non_transpo_points_Mandaluyong, graph_of_stops_Mandaluyong, list_of_stops_Mandaluyong)

# ASSIGN POPULATION AND INTERMODAL ATTRIBUTES
list_of_stops_Mandaluyong, graph_of_stops_Mandaluyong = (
    assign_existing_stop_population_attributes(
        list_of_stops_Mandaluyong,
        graph_of_stops_Mandaluyong,
        mandaluyong_zone_graph
    )
)

# ASSIGN FLOOD ATTRIBUTES
assign_flood_attributes(
    list_of_stops_Mandaluyong,
    graph_of_stops_Mandaluyong,
    flood_5_gdf,
    flood_25_gdf,
    flood_100_gdf
)

validate_existing_stop_attributes(
    list_of_stops_Mandaluyong,
    graph_of_stops_Mandaluyong
)


# Visualize the stops
map = plot_stops_on_map(list_of_stops_Mandaluyong)
map.save("stop_map_mandaluyong.html")

In [76]:
#Export stops to pickle
file_path = f'{pikl_filepath}stop_list_Mandaluyong.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(list_of_stops_Mandaluyong, f)
    
file_path = f'{pikl_filepath}stop_graph_Mandaluyong.pkl'
with open(file_path, 'wb') as f:
    pickle.dump(graph_of_stops_Mandaluyong, f)

### Creating the routes

In [156]:
# Manila
route_network_Manila, route_network_graph_Manila, reverse_route_network_Manila = generate_route_network(route_list_Manila, graph_of_stops_Manila)
assign_dynamic_population_scores(
    route_network_graph_Manila,
    ZONE_POPULATION_MANILA,
    POPULATION_DENOMINATOR_MANILA
)
validate_network_population_scores(
    route_network_graph_Manila,
    ZONE_POPULATION_MANILA,
    POPULATION_DENOMINATOR_MANILA
)
Manila_Route_Network = networkObj(
    route_network_Manila,
    route_network_graph_Manila,
    reversed_routes=reverse_route_network_Manila
)

0 - 1
1 - 835
835 - 836
836 - 4
4 - 839
839 - 6
6 - 7
7 - 8
8 - 9
9 - 843
843 - 11
11 - 12

13 - 14
14 - 15
15 - 17
17 - 18
18 - 19
19 - 20
20 - 21
21 - 22
22 - 23
23 - 24
24 - 25
25 - 27
27 - 28
28 - 31
31 - 32
32 - 33

34 - 35
35 - 36
36 - 37
37 - 38
38 - 39

40 - 42
42 - 45
45 - 46
46 - 47
47 - 48
48 - 49
49 - 50
50 - 51
51 - 234
234 - 53
53 - 57

64 - 65
65 - 66
66 - 67
67 - 68
68 - 70
70 - 71
71 - 73
73 - 76
76 - 78
78 - 79

21 - 22
22 - 82
82 - 23
23 - 84
84 - 24
24 - 25
25 - 27
27 - 92
92 - 93

94 - 95
95 - 96
96 - 97
97 - 98
98 - 102
102 - 103
103 - 104
104 - 105
105 - 106
106 - 107
107 - 108

109 - 110
110 - 111
111 - 432
432 - 113

114 - 115
115 - 116
116 - 117
117 - 118
118 - 119
119 - 120

121 - 122
122 - 123
123 - 124
124 - 125
125 - 126
126 - 127
127 - 128
128 - 129
129 - 116
116 - 131
131 - 119
119 - 120
120 - 134
134 - 135

22 - 21
21 - 138
138 - 139
139 - 46
46 - 47
47 - 48
48 - 50
50 - 145
145 - 146
146 - 147
147 - 148
148 - 149
149 - 150
150 - 151
151 - 935
935 - 153

In [157]:
print("--- MANILA EXISTING ROUTE VALIDATION ---")

forward_count = len(
    Manila_Route_Network.routes
)

reverse_count = sum(
    route is not None
    for route in Manila_Route_Network.reversed_routes
)

missing_reverse = [
    route.route_id
    for route, reverse in zip(
        Manila_Route_Network.routes,
        Manila_Route_Network.reversed_routes
    )
    if reverse is None
]

print("Forward routes:", forward_count)
print("Available reverse routes:", reverse_count)
print("Missing reverse routes:", len(missing_reverse))
print("Routes without reverse service:", missing_reverse)

assert (
    len(Manila_Route_Network.routes)
    == len(Manila_Route_Network.reversed_routes)
)

missing_source_ids = [
    node_id
    for node_id, data
    in Manila_Route_Network.graph.nodes(data=True)
    if (
        data.get("isTranspo", False)
        and data.get("transport_source_id") is None
    )
]

print(
    "Intermodal stops missing source IDs:",
    len(missing_source_ids)
)

assert not missing_source_ids

print("Existing route representation validation passed.")

--- MANILA EXISTING ROUTE VALIDATION ---
Forward routes: 88
Available reverse routes: 86
Missing reverse routes: 2
Routes without reverse service: ['11-A', '33-A']
Intermodal stops missing source IDs: 0
Existing route representation validation passed.


In [159]:
def diagnose_missing_reverse_routes(
    network,
    graph_of_stops
):
    for forward_route, reverse_route in zip(
        network.routes,
        network.reversed_routes
    ):

        if reverse_route is not None:
            continue

        print("\n" + "=" * 55)
        print("Forward route:", forward_route.route_id)
        print("=" * 55)

        # Recover the original forward stop sequence.
        forward_stops = [
            forward_route.route[0][0]
        ] + [
            connection[-1]
            for connection in forward_route.route
        ]

        reverse_stops = forward_stops[::-1]

        failed_connections = 0

        for index, (start, end) in enumerate(
            zip(
                reverse_stops[:-1],
                reverse_stops[1:]
            ),
            start=1
        ):

            if nx.has_path(
                graph_of_stops,
                start,
                end
            ):
                continue

            failed_connections += 1

            start_data = graph_of_stops.nodes[start]
            end_data = graph_of_stops.nodes[end]

            print(
                f"\nFailed reverse connection {index}: "
                f"{start} -> {end}"
            )

            print(
                "Start coordinates:",
                (
                    start_data["y"],
                    start_data["x"]
                )
            )

            print(
                "End coordinates:",
                (
                    end_data["y"],
                    end_data["x"]
                )
            )

            print(
                "Start in-degree/out-degree:",
                graph_of_stops.in_degree(start),
                graph_of_stops.out_degree(start)
            )

            print(
                "End in-degree/out-degree:",
                graph_of_stops.in_degree(end),
                graph_of_stops.out_degree(end)
            )

            print(
                "Opposite connection exists:",
                nx.has_path(
                    graph_of_stops,
                    end,
                    start
                )
            )

        print(
            "\nTotal failed reverse connections:",
            failed_connections
        )


diagnose_missing_reverse_routes(
    Manila_Route_Network,
    graph_of_stops_Manila
)


Forward route: 11-A

Failed reverse connection 1: 166 -> 165
Start coordinates: (14.625491119774669, 120.96016603173257)
End coordinates: (14.623180796574625, 120.9600848950972)
Start in-degree/out-degree: 1 1
End in-degree/out-degree: 1 1
Opposite connection exists: True

Total failed reverse connections: 1

Forward route: 33-A

Failed reverse connection 9: 426 -> 425
Start coordinates: (14.619677320046035, 120.9597255543281)
End coordinates: (14.622967895352447, 120.9598832792467)
Start in-degree/out-degree: 1 1
End in-degree/out-degree: 1 1
Opposite connection exists: True

Failed reverse connection 10: 425 -> 424
Start coordinates: (14.622967895352447, 120.9598832792467)
End coordinates: (14.625485763970802, 120.95998300579753)
Start in-degree/out-degree: 1 1
End in-degree/out-degree: 1 1
Opposite connection exists: True

Total failed reverse connections: 2


In [161]:
# =========================================================
# DIAGNOSE OPPOSITE-DIRECTION PARTNER EDGES
# =========================================================

PARTNER_EDGE_MAX_DISTANCE_M = 50
PARTNER_DIRECTION_COS_THRESHOLD = -0.5

# Required by the GA's geometry helper.
CITY_GRAPH = MANILA_GRAPH

load_ga_function_definitions([
    "is_oneway_value",
    "highway_is_eligible",
    "normalize_road_names",
    "get_original_edge_geometry",
    "get_edge_direction_vector",
    "build_partner_edge_map"
])

# Reuse the same eligible road reference as the GA.
partner_edges_Manila = build_partner_edge_map(
    manila_road[
        manila_road["junction"].isna()
    ]
)

road_data_by_id = {
    int(row["edge_id"]): row
    for _, row in manila_road.iterrows()
}

failed_connections = {
    "11-A": [(166, 165)],
    "33-A": [(426, 425), (425, 424)]
}

affected_stops = sorted({
    stop_id
    for pairs in failed_connections.values()
    for pair in pairs
    for stop_id in pair
})

for stop_id in affected_stops:

    stop_data = graph_of_stops_Manila.nodes[
        stop_id
    ]

    source_edge_id = stop_data.get(
        "source_edge_id"
    )

    if source_edge_id is None:
        print(f"\nStop {stop_id}: no source edge")
        continue

    source_edge_id = int(source_edge_id)

    road_data = road_data_by_id[
        source_edge_id
    ]

    partner_id = partner_edges_Manila.get(
        source_edge_id
    )

    print("\n" + "=" * 50)
    print("Stop:", stop_id)
    print("Source edge:", source_edge_id)
    print("Road name:", road_data.get("name"))
    print("One-way:", road_data.get("oneway"))
    print("Highway:", road_data.get("highway"))
    print("Partner edge:", partner_id)

    if partner_id is None:
        print("No opposite-direction partner identified.")
        continue

    # Find existing stops snapped onto that partner edge.
    candidates = []

    for stop in list_of_stops_Manila:

        candidate_edge = (
            graph_of_stops_Manila.nodes[
                stop.id
            ].get("source_edge_id")
        )

        if candidate_edge is None:
            continue

        if int(candidate_edge) != partner_id:
            continue

        distance_m = geodesic(
            (stop_data["y"], stop_data["x"]),
            (stop.lat, stop.long)
        ).meters

        candidates.append(
            (stop.id, round(distance_m, 2))
        )

    candidates.sort(
        key=lambda item: item[1]
    )

    print(
        "Nearest existing stops on partner edge:",
        candidates[:5]
    )

# Check whether the failed connections are
# physically connected when direction is ignored.
undirected_road_graph = (
    graph_of_stops_Manila.to_undirected(
        as_view=True
    )
)

print("\n=== FAILED CONNECTION REACHABILITY ===")

for route_id, pairs in failed_connections.items():

    for start, end in pairs:

        print(
            route_id,
            f"{start} -> {end}",
            "| Undirected path:",
            nx.has_path(
                undirected_road_graph,
                start,
                end
            )
        )

Loaded GA functions: is_oneway_value, highway_is_eligible, normalize_road_names, get_original_edge_geometry, get_edge_direction_vector, build_partner_edge_map
Opposite-direction edge pairs: 456

Stop: 165
Source edge: 5654
Road name: Mel Lopez Boulevard
One-way: True
Highway: trunk
Partner edge: None
No opposite-direction partner identified.

Stop: 166
Source edge: 476
Road name: Mel Lopez Boulevard
One-way: True
Highway: trunk
Partner edge: 7651
Nearest existing stops on partner edge: [(424, 19.73), (425, 280.84)]

Stop: 424
Source edge: 7651
Road name: Mel Lopez Boulevard
One-way: True
Highway: trunk
Partner edge: 476
Nearest existing stops on partner edge: [(166, 19.73)]

Stop: 425
Source edge: 7651
Road name: Mel Lopez Boulevard
One-way: True
Highway: trunk
Partner edge: 476
Nearest existing stops on partner edge: [(166, 280.84)]

Stop: 426
Source edge: 3344
Road name: Mel Lopez Boulevard
One-way: True
Highway: trunk
Partner edge: 7528
Nearest existing stops on partner edge: []

==

In [162]:
def get_forward_stop_ids(route):
    return (
        [route.route[0][0]]
        + [path[-1] for path in route.route]
    )


route_11 = next(
    route for route in Manila_Route_Network.routes
    if route.route_id == "11-A"
)

route_33 = next(
    route for route in Manila_Route_Network.routes
    if route.route_id == "33-A"
)

stops_11 = get_forward_stop_ids(route_11)
stops_33 = get_forward_stop_ids(route_33)

print("Route 11 forward stops:", len(stops_11))
print("Route 33 forward stops:", len(stops_33))

print("Route 11 distance (km):", route_11.length)
print("Route 33 distance (km):", route_33.length)

matches = []

for i, stop_11 in enumerate(stops_11):
    data_11 = graph_of_stops_Manila.nodes[stop_11]

    for j, stop_33 in enumerate(stops_33):
        data_33 = graph_of_stops_Manila.nodes[stop_33]

        distance_m = geodesic(
            (data_11["y"], data_11["x"]),
            (data_33["y"], data_33["x"])
        ).meters

        if distance_m <= 50:
            matches.append({
                "Route 11 Position": i,
                "Route 11 Stop": stop_11,
                "Route 33 Position": j,
                "Route 33 Stop": stop_33,
                "Distance (m)": round(distance_m, 2)
            })

matches_df = pd.DataFrame(matches)

print("\nMatched nearby stop pairs:", len(matches_df))

display(matches_df)

print(
    "\n424 -> 425 is legally reachable:",
    nx.has_path(graph_of_stops_Manila, 424, 425)
)

print(
    "165 -> 166 is legally reachable:",
    nx.has_path(graph_of_stops_Manila, 165, 166)
)

Route 11 forward stops: 9
Route 33 forward stops: 11
Route 11 distance (km): 2.6364362928183622
Route 33 distance (km): 2.7139135083356662

Matched nearby stop pairs: 6


,Route 11 Position,Route 11 Stop,Route 33 Position,Route 33 Stop,Distance (m)
0,0,158,10,434,14.11
1,2,160,6,430,17.32
2,4,162,4,428,20.59
3,5,163,3,427,25.82
4,7,165,1,425,32.04
5,8,166,0,424,19.73



424 -> 425 is legally reachable: True
165 -> 166 is legally reachable: True


In [163]:
# =========================================================
# IDENTIFY ORIGINAL SOURCE ROUTE LABELS
# =========================================================

# generate_route_network() filters routes containing
# fewer than four stops after checking directed reachability.

valid_original_indices = [
    index
    for index, route in enumerate(route_list_Manila)
    if len(route) > 3
]

assert len(valid_original_indices) == len(
    Manila_Route_Network.routes
)

for route_number in [11, 33]:

    route = Manila_Route_Network.routes[
        route_number
    ]

    original_index = valid_original_indices[
        route_number
    ]

    source_route = manila_route_stops_new[
        original_index
    ]

    original_route_names = sorted({
        str(name)
        for _, name in source_route
    })

    print("\n" + "=" * 50)
    print("Generated Route ID:", route.route_id)
    print("Original Route Index:", original_index)
    print("Original Route Name(s):",
          original_route_names)
    print("Original Stop Count:",
          len(source_route))
    print("Reconstructed Stop Count:",
          len(route.route) + 1)
    print("Route Distance (km):",
          route.length)


Generated Route ID: 11-A
Original Route Index: 11
Original Route Name(s): ['16']
Original Stop Count: 9
Reconstructed Stop Count: 9
Route Distance (km): 2.6364362928183622

Generated Route ID: 33-A
Original Route Index: 33
Original Route Name(s): ['48']
Original Stop Count: 11
Reconstructed Stop Count: 11
Route Distance (km): 2.7139135083356662


In [ ]:
# MAKATI ROUTES
route_network_Makati, route_network_graph_Makati, reverse_route_network_Makati = generate_route_network(route_list_Makati, graph_of_stops_Makati)
assign_dynamic_population_scores(
    route_network_graph_Makati,
    ZONE_POPULATION_MAKATI,
    POPULATION_DENOMINATOR_MAKATI
)
validate_network_population_scores(
    route_network_graph_Makati,
    ZONE_POPULATION_MAKATI,
    POPULATION_DENOMINATOR_MAKATI
)
Makati_Route_Network = networkObj(route_network_Makati, route_network_graph_Makati)

0 - 1
1 - 2
2 - 3

4 - 5
5 - 75
75 - 7
7 - 8

9 - 10
10 - 62
62 - 63
63 - 64
64 - 15

15 - 16
16 - 17
17 - 18
18 - 19
19 - 20
20 - 21
21 - 22
22 - 23
23 - 24

25 - 26
26 - 27
27 - 28
28 - 29
29 - 30
30 - 31
31 - 32
32 - 33
33 - 34
34 - 172
172 - 36

37 - 81
81 - 83
83 - 5
5 - 7
7 - 75

3 - 44
44 - 45
45 - 29
29 - 47
47 - 48

49 - 50
50 - 51
51 - 52
52 - 198

84 - 55
55 - 56
56 - 195
195 - 58
58 - 59

24 - 10
10 - 62
62 - 63
63 - 64
64 - 15

109 - 67
67 - 111
111 - 69
69 - 70
70 - 71
71 - 72
72 - 82
82 - 5
5 - 75
75 - 76

77 - 78
78 - 79
79 - 80
80 - 81
81 - 82
82 - 83
83 - 84
84 - 198

86 - 87
87 - 153
153 - 154
154 - 155
155 - 156
156 - 92
92 - 93
93 - 94
94 - 95
95 - 184
184 - 97
97 - 98
98 - 187
187 - 100
100 - 189
189 - 102
102 - 103

104 - 199
199 - 200
200 - 107
107 - 108

109 - 67
67 - 111
111 - 69
69 - 70
70 - 71
71 - 115
115 - 116
116 - 135

198 - 84
84 - 133
133 - 83

122 - 123
123 - 124
124 - 125
125 - 126
126 - 127
127 - 128
128 - 115

76 - 75
75 - 132
132 - 133
133 - 134



In [ ]:
# MANDALUYONG
route_network_Mandaluyong, route_network_graph_Mandaluyong, reverse_route_network_Mandaluyong = generate_route_network(route_list_Mandaluyong, graph_of_stops_Mandaluyong)
assign_dynamic_population_scores(
    route_network_graph_Mandaluyong,
    ZONE_POPULATION_MANDALUYONG,
    POPULATION_DENOMINATOR_MANDALUYONG
)
validate_network_population_scores(
    route_network_graph_Mandaluyong,
    ZONE_POPULATION_MANDALUYONG,
    POPULATION_DENOMINATOR_MANDALUYONG
)
Mandaluyong_Route_Network = networkObj(route_network_Mandaluyong, route_network_graph_Mandaluyong)

0 - 12
12 - 13
13 - 3
3 - 97
97 - 16
16 - 89
89 - 7
7 - 8
8 - 36

10 - 0
0 - 12
12 - 13
13 - 3
3 - 97
97 - 16
16 - 89
89 - 7
7 - 36

56 - 47
47 - 104
104 - 23
23 - 24
24 - 25
25 - 26
26 - 27
27 - 28
28 - 29
29 - 30
30 - 31
31 - 32

33 - 56
56 - 35
35 - 36
36 - 37

47 - 104
104 - 23
23 - 50
50 - 51
51 - 52
52 - 53
53 - 54
54 - 55

56 - 57
57 - 58
58 - 59
59 - 60
60 - 61
61 - 62
62 - 63
63 - 64
64 - 89
89 - 66
66 - 16
16 - 97
97 - 69
69 - 70
70 - 71
71 - 72
72 - 73
73 - 74
74 - 12
12 - 0
0 - 77

78 - 79
79 - 80
80 - 81
81 - 82
82 - 83
83 - 84
84 - 85
85 - 86
86 - 87
87 - 88
88 - 89
89 - 64
64 - 36
36 - 56

0 - 12
12 - 13
13 - 3
3 - 97
97 - 16
16 - 99
99 - 100
100 - 24
24 - 50
50 - 23
23 - 104
104 - 47
47 - 10


Real route
0 - 12
12 - 13
13 - 3
3 - 97
97 - 16
16 - 89
89 - 7
7 - 8
8 - 36

[[36, 8], [8, 7], [7, 89], [89, 16], [16, 97], [97, 3], [3, 13], [13, 12], [12, 0]]
10 - 0
0 - 12
12 - 13
13 - 3
3 - 97
97 - 16
16 - 89
89 - 7
7 - 36

[[36, 7], [7, 89], [89, 16], [16, 97], [97, 3], [3, 1

### Import Existing Stops and Networks

In [49]:
# Stops
with open(f'{pikl_filepath}stop_graph_Manila_Thesis_Baseline.pkl', 'rb') as f:
    graph_of_stops_Manila = pickle.load(f)
#with open(f'{pikl_filepath}stop_graph_Makati.pkl', 'rb') as f:
    #graph_of_stops_Makati = pickle.load(f)
#with open(f'{pikl_filepath}stop_graph_Mandaluyong.pkl', 'rb') as f:
    #graph_of_stops_Mandaluyong = pickle.load(f)

# Route Networks
Manila_Route_Network = import_networks(f'{pikl_filepath}Manila Route Network Thesis Baseline.pikl')
#Makati_Route_Network = import_networks(f'{pikl_filepath}Makati Route Network.pikl')
#Mandaluyong_Route_Network = import_networks(f'{pikl_filepath}Mandaluyong Route Network.pikl')

### Exporting Route Networks

In [158]:
#pikl files

export_networks(
    Manila_Route_Network,
    f"{pikl_filepath}"
    "Manila Route Network Thesis Baseline.pikl"
)
export_networks(
    graph_of_stops_Manila,
    f"{pikl_filepath}"
    "stop_graph_Manila_Thesis_Baseline.pkl"
)

#export_networks(Makati_Route_Network, f'{pikl_filepath}Makati Route Network.pikl')

#export_networks(Mandaluyong_Route_Network, f'{pikl_filepath}Mandaluyong Route Network.pikl')

In [160]:
save_route_map_html(
    Manila_Route_Network,
    graph_of_stops_Manila,
    f"{map_filepath}Manila Route Map.html"
)

#save_route_map_html(
#    Makati_Route_Network,
#    graph_of_stops_Makati,
#    f"{map_filepath}Makati Route Map.html"
#)

#save_route_map_html(
#    Mandaluyong_Route_Network,
#    graph_of_stops_Mandaluyong,
#    f"{map_filepath}Mandaluyong Route Map.html"
#)

Saved HTML map to: Original Jeepney Routes/Maps/Manila Route Map.html


### Network Metrics Result

In [ ]:
def print_city_statistics(
    network,
    avg_betweenness,
    avg_closeness,
    longest_route,
    long_stops,
    shortest_route,
    short_stops,
    avg_route_length,
    avg_degree,
    avg_stops,
    total_road_distance,
    graph_of_stops,
    total_potential_transport_nodes,
    min_intersections_per_route,
    min_transpo_stops_per_route,
    weight_coverage,
    weight_internal_connectivity,
    weight_external_connectivity,
    weight_vulnerability,
    weight_robustness
):
    fitness_score, components = compute_fitness_score(
        network.graph,
        weight_coverage,
        weight_internal_connectivity,
        weight_external_connectivity,
        weight_vulnerability,
        weight_robustness,
        total_road_distance,
        graph_of_stops,
        total_potential_transport_nodes,
        min_intersections_per_route,
        min_transpo_stops_per_route
    )

    print(f"Coverage Score: {components['Coverage Raw']:.4f}")
    print(f"Internal Connectivity Score: {components['Internal Connectivity Raw']:.4f}")
    print(f"External Connectivity Score: {components['External Connectivity Raw']:.4f}")
    print(f"Vulnerability Score: {components['Vulnerability Raw']:.4f}")
    print(f"Robustness Score: {components['Robustness Raw']:.4f}")
    print(f"Fitness Score: {fitness_score:.4f}")
    print(f"Total Routes: {len(network.routes)}")
    print(f"Total Stops: {len(network.graph.nodes(data=True))}")
    print(f"Total Network Nodes: {components['Total Network Nodes']}")
    print(f"Total Potential Transport Nodes: {components['Total Potential Transport Nodes']}")
    print(f"Shortest Route: {shortest_route:.4f}")
    print(f"Longest Route: {longest_route:.4f}")
    print(f"Average Route Length: {avg_route_length:.4f}")
    print(f"Average Degree: {avg_degree:.4f}")
    print(f"Average Number of Stops per Route: {avg_stops:.4f}")
    print(f"Diameter: {get_network_diameter(network.graph):.4f}")
    print(f"Max Betweenness Centrality: {avg_betweenness:.4f}")
    print(f"Max Closeness Centrality: {avg_closeness:.4f}")
    print(f"n_i: {components['n_i']}")
    print(f"N_i: {components['N_i']}")
    print(f"r_i: {components['r_i']}")
    print(f"R_i: {components['R_i']}")
    print(f"Min Intersections Per Route: {components['Min Intersections Per Route']}")
    print(f"n_e: {components['n_e']}")
    print(f"N_e: {components['N_e']}")
    print(f"r_e: {components['r_e']}")
    print(f"R_e: {components['R_e']}")
    print(f"Min Transpo Stops Per Route: {components['Min Transpo Stops Per Route']}")
    print(f"Target Node Ratio: {components['Target Node Ratio']:.2f}")
    print(f"q_c Vulnerability: {components['q_c Vulnerability']:.4f}")
    print(f"q_c Robustness: {components['q_c Robustness']:.4f}")
    print("")

In [ ]:
_, betweenness_centrality_Manila, closeness_centrality_Manila = get_node_centrality(Manila_Route_Network.graph)
# avg_betweenness_centrality_Manila = sum(betweenness_centrality_Manila.values()) / len(betweenness_centrality_Manila)
# avg_closeness_centrality_Manila = sum(closeness_centrality_Manila.values()) / len(closeness_centrality_Manila)
max_betweenness_centrality_Manila = max(betweenness_centrality_Manila.values())
max_closeness_centrality_Manila = max(closeness_centrality_Manila.values())
longest_route_Manila, long_route_stops_Manila = get_longest_route(Manila_Route_Network.routes)
shortest_route_Manila, short_route_stops_Manila = get_shortest_route(Manila_Route_Network.routes)
avg_route_length_Manila = get_average_route_length(Manila_Route_Network.routes)
avg_degree_Manila = average_degree(Manila_Route_Network.graph)
avg_number_stops_Manila = average_number_of_stops_per_route(Manila_Route_Network.routes)
manila_connectivity_config = CITY_CONNECTIVITY_CONFIG["Manila"]

manila_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}

print("Manila Existing Jeepney Network Statistics")
print_city_statistics(
    Manila_Route_Network,
    max_betweenness_centrality_Manila,
    max_closeness_centrality_Manila,
    longest_route_Manila.length,
    long_route_stops_Manila,
    shortest_route_Manila.length,
    short_route_stops_Manila,
    avg_route_length_Manila,
    avg_degree_Manila,
    avg_number_stops_Manila,
    TOTAL_NETWORK_ROAD_DISTANCE_MANILA,
    graph_of_stops_Manila,
    total_potential_transport_nodes_Manila,
    manila_connectivity_config["min_intersections_per_route"],
    manila_connectivity_config["min_transpo_stops_per_route"],
    #route_list_Manila, # This is for the route list that will be used for the coverage score
    manila_weights["weight_coverage"],
    manila_weights["weight_internal_connectivity"],
    manila_weights["weight_external_connectivity"],
    manila_weights["weight_vulnerability"],
    manila_weights["weight_robustness"]
)

Manila Existing Jeepney Network Statistics
Coverage Score: 0.4515
Internal Connectivity Score: 0.7306
External Connectivity Score: 0.1786
Vulnerability Score: 0.3504
Robustness Score: 0.6590
Fitness Score: 0.4740
Total Routes: 88
Total Stops: 394
Total Network Nodes: 394
Total Potential Transport Nodes: 464
Shortest Route: 1.0571
Longest Route: 37.1687
Average Route Length: 14.3749
Average Degree: 9.4213
Average Number of Stops per Route: 11.6477
Diameter: 38.7395
Max Betweenness Centrality: 0.2237
Max Closeness Centrality: 0.1307
n_i: 183
N_i: 295
r_i: 74
R_i: 88
Min Intersections Per Route: 4
n_e: 55
N_e: 464
r_e: 21
R_e: 88
Min Transpo Stops Per Route: 3
Target Node Ratio: 0.75
q_c Vulnerability: 0.0990
q_c Robustness: 0.0990



In [ ]:
_, betweenness_centrality_Makati, closeness_centrality_Makati = get_node_centrality(Makati_Route_Network.graph)
# avg_betweenness_centrality_Makati = sum(betweenness_centrality_Makati.values()) / len(betweenness_centrality_Makati)
# avg_closeness_centrality_Makati = sum(closeness_centrality_Makati.values()) / len(closeness_centrality_Makati)
max_betweenness_centrality_Makati = max(betweenness_centrality_Makati.values())
max_closeness_centrality_Makati = max(closeness_centrality_Makati.values())
longest_route_Makati, long_route_stops_Makati = get_longest_route(Makati_Route_Network.routes)
shortest_route_Makati, short_route_stops_Makati = get_shortest_route(Makati_Route_Network.routes)
avg_route_length_Makati = get_average_route_length(Makati_Route_Network.routes)
avg_degree_Makati = average_degree(Makati_Route_Network.graph)
avg_number_stops_Makati = average_number_of_stops_per_route(Makati_Route_Network.routes)
makati_connectivity_config = CITY_CONNECTIVITY_CONFIG["Makati"]

makati_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}

print("Makati Existing Jeepney Network Statistics")
print_city_statistics(
    Makati_Route_Network,
    max_betweenness_centrality_Makati,
    max_closeness_centrality_Makati,
    longest_route_Makati.length,
    long_route_stops_Makati,
    shortest_route_Makati.length,
    short_route_stops_Makati,
    avg_route_length_Makati,
    avg_degree_Makati,
    avg_number_stops_Makati,
    TOTAL_NETWORK_ROAD_DISTANCE_MAKATI,
    graph_of_stops_Makati,
    total_potential_transport_nodes_Makati,
    makati_connectivity_config["min_intersections_per_route"],
    makati_connectivity_config["min_transpo_stops_per_route"],
    # route_list_Makati, # This is for the route list that will be used for the coverage score
    makati_weights["weight_coverage"],
    makati_weights["weight_internal_connectivity"],
    makati_weights["weight_external_connectivity"],
    makati_weights["weight_vulnerability"],
    makati_weights["weight_robustness"]
)

Makati Existing Jeepney Network Statistics
Coverage Score: 0.3030
Internal Connectivity Score: 0.7573
External Connectivity Score: 0.3912
Vulnerability Score: 0.6763
Robustness Score: 0.9098
Fitness Score: 0.6075
Total Routes: 25
Total Stops: 139
Total Network Nodes: 139
Total Potential Transport Nodes: 111
Shortest Route: 1.0147
Longest Route: 11.8452
Average Route Length: 4.7568
Average Degree: 6.1583
Average Number of Stops per Route: 9.5600
Diameter: 22.3997
Max Betweenness Centrality: 0.4301
Max Closeness Centrality: 0.1293
n_i: 66
N_i: 104
r_i: 22
R_i: 25
Min Intersections Per Route: 3
n_e: 38
N_e: 111
r_e: 11
R_e: 25
Min Transpo Stops Per Route: 2
Target Node Ratio: 0.75
q_c Vulnerability: 0.0935
q_c Robustness: 0.0935



In [ ]:
_, betweenness_centrality_Mandaluyong, closeness_centrality_Mandaluyong = get_node_centrality(Mandaluyong_Route_Network.graph)
# avg_betweenness_centrality_Mandaluyong = sum(betweenness_centrality_Mandaluyong.values()) / len(betweenness_centrality_Mandaluyong)
# avg_closeness_centrality_Mandaluyong = sum(closeness_centrality_Mandaluyong.values()) / len(closeness_centrality_Mandaluyong)
max_betweenness_centrality_Mandaluyong = max(betweenness_centrality_Mandaluyong.values())
max_closeness_centrality_Mandaluyong = max(closeness_centrality_Mandaluyong.values())
longest_route_Mandaluyong, long_route_stops_Mandaluyong = get_longest_route(Mandaluyong_Route_Network.routes)
shortest_route_Mandaluyong, short_route_stops_Mandaluyong = get_shortest_route(Mandaluyong_Route_Network.routes)
avg_route_length_Mandaluyong = get_average_route_length(Mandaluyong_Route_Network.routes)
avg_degree_Mandaluyong = average_degree(Mandaluyong_Route_Network.graph)
avg_number_stops_Mandaluyong = average_number_of_stops_per_route(Mandaluyong_Route_Network.routes)
mandaluyong_connectivity_config = CITY_CONNECTIVITY_CONFIG["Mandaluyong"]

mandaluyong_weights = {
    "weight_coverage": 0.20,
    "weight_internal_connectivity": 0.20,
    "weight_external_connectivity": 0.20,
    "weight_vulnerability": 0.20,
    "weight_robustness": 0.20
}


print("Mandaluyong Existing Jeepney Network Statistics")
print_city_statistics(
    Mandaluyong_Route_Network,
    max_betweenness_centrality_Mandaluyong,
    max_closeness_centrality_Mandaluyong,
    longest_route_Mandaluyong.length,
    long_route_stops_Mandaluyong,
    shortest_route_Mandaluyong.length,
    short_route_stops_Mandaluyong,
    avg_route_length_Mandaluyong,
    avg_degree_Mandaluyong,
    avg_number_stops_Mandaluyong,
    TOTAL_NETWORK_ROAD_DISTANCE_MANDALUYONG,
    graph_of_stops_Mandaluyong,
    total_potential_transport_nodes_Mandaluyong,
    mandaluyong_connectivity_config["min_intersections_per_route"],
    mandaluyong_connectivity_config["min_transpo_stops_per_route"],
    # route_list_Mandaluyong, # This is for the route list that will be used for the coverage score
    mandaluyong_weights["weight_coverage"],
    mandaluyong_weights["weight_internal_connectivity"],
    mandaluyong_weights["weight_external_connectivity"],
    mandaluyong_weights["weight_vulnerability"],
    mandaluyong_weights["weight_robustness"]
)

Mandaluyong Existing Jeepney Network Statistics
Coverage Score: 0.1955
Internal Connectivity Score: 0.6848
External Connectivity Score: 0.6111
Vulnerability Score: 0.5591
Robustness Score: 0.8495
Fitness Score: 0.5800
Total Routes: 8
Total Stops: 62
Total Network Nodes: 62
Total Potential Transport Nodes: 45
Shortest Route: 3.5106
Longest Route: 9.9029
Average Route Length: 5.6606
Average Degree: 5.6774
Average Number of Stops per Route: 12.2500
Diameter: 13.1033
Max Betweenness Centrality: 0.4111
Max Closeness Centrality: 0.1910
n_i: 17
N_i: 46
r_i: 8
R_i: 8
Min Intersections Per Route: 2
n_e: 10
N_e: 45
r_e: 8
R_e: 8
Min Transpo Stops Per Route: 1
Target Node Ratio: 0.75
q_c Vulnerability: 0.0968
q_c Robustness: 0.0968



### Manila Thesis Baseline Evaluation

In [50]:
from scipy.special import expit

# =========================================================
# SHARED MANILA FITNESS CONFIGURATION
# =========================================================

MIN_INTERMODAL_STOPS_PER_ROUTE = 3

ROBUSTNESS_REMOVAL_FRACTION = 0.10

HEADWAY_TARGET_SECONDS = 360.0
HEADWAY_CONGESTION_FACTOR = 0.57
HEADWAY_DWELL_MIN_SECONDS = 20.0
HEADWAY_DWELL_ADDITIONAL_SECONDS = 40.0

SEED = 42

# Identical road-class eligibility.
filter_options = [
    "primary",
    "secondary",
    "tertiary",
    "trunk"
]

# Load the GA's actual implementation.
load_ga_function_definitions([
    "route_signature",
    "get_route_stop_sequence",

    "assign_dynamic_population_scores",

    "get_route_family_stop_map",
    "prepare_fitness_static_cache",

    "get_path_edge_record",
    "get_path_edge_data",
    "prepare_coverage_edge_cache",
    "compute_coverage",

    "compute_connectivity",

    "get_robustness_removal_count",
    "build_weak_neighbor_map",
    "compute_lwcc_retention_score",
    "get_seeded_node_order",
    "compute_targeted_attack_robustness",
    "compute_random_attack_robustness",
    "compute_flood_robustness",

    "prepare_headway_edge_times",
    "compute_headway_dwell_time",
    "get_headway_route_static_data",
    "compute_route_direction_time",
    "compute_average_required_jeepneys",
    "compute_headway_efficiency_from_average",

    "validate_fitness_metrics",
    "compute_fitness_score"
])

print("Shared GA fitness functions loaded.")

Loaded GA functions: route_signature, get_route_stop_sequence, assign_dynamic_population_scores, get_route_family_stop_map, prepare_fitness_static_cache, get_path_edge_record, get_path_edge_data, prepare_coverage_edge_cache, compute_coverage, compute_connectivity, get_robustness_removal_count, build_weak_neighbor_map, compute_lwcc_retention_score, get_seeded_node_order, compute_targeted_attack_robustness, compute_random_attack_robustness, compute_flood_robustness, prepare_headway_edge_times, compute_headway_dwell_time, get_headway_route_static_data, compute_route_direction_time, compute_average_required_jeepneys, compute_headway_efficiency_from_average, validate_fitness_metrics, compute_fitness_score
Shared GA fitness functions loaded.


In [51]:
# =========================================================
# MANILA COVERAGE REFERENCE
# =========================================================

def highway_is_eligible(value):

    if isinstance(
        value,
        (list, tuple, set, np.ndarray)
    ):
        return any(
            highway in filter_options
            for highway in value
        )

    return value in filter_options


coverage_road_gdf = ox.graph_to_gdfs(
    MANILA_GRAPH,
    nodes=False,
    edges=True
)

COVERAGE_ELIGIBLE_ROADS = coverage_road_gdf.loc[
    coverage_road_gdf["highway"].apply(
        highway_is_eligible
    )
].copy()

COVERAGE_DENOMINATOR_EDGE_IDS = {
    int(edge_id)
    for edge_id in (
        COVERAGE_ELIGIBLE_ROADS["edge_id"]
    )
}

assert (
    len(COVERAGE_DENOMINATOR_EDGE_IDS)
    == len(COVERAGE_ELIGIBLE_ROADS)
), "Duplicate coverage edge IDs."

TOTAL_NETWORK_ROAD_DISTANCE = float(
    COVERAGE_ELIGIBLE_ROADS["length"].sum()
)

print(
    "Manila eligible road length (km):",
    TOTAL_NETWORK_ROAD_DISTANCE / 1000
)

print(
    "Eligible directed road edges:",
    len(COVERAGE_DENOMINATOR_EDGE_IDS)
)

Manila eligible road length (km): 372.43920670656996
Eligible directed road edges: 4345


In [53]:
# =========================================================
# IMPORT THE MATCHING GA RESULT
# =========================================================

GA_RESULT_FILE = (
    "Saved Networks/Manila/"
    "Manila_GA_Route_networks_Mixed_"
    "cov17_k17_t17_r17_b17_h17_S42.pkl"
)

ga_results = import_networks(
    GA_RESULT_FILE
)

optimized_network = ga_results["population"][0]

SAVED_OPTIMIZED_FITNESS = float(
    optimized_network.fitness_score
)

HEADWAY_BASELINE_MEAN_MANILA = float(
    ga_results["headway_normalization"]["baseline_mean"]
)

HEADWAY_BASELINE_STD_MANILA = float(
    ga_results["headway_normalization"]["baseline_std"]
)

print(
    "Headway baseline mean:",
    HEADWAY_BASELINE_MEAN_MANILA
)

print(
    "Headway baseline std:",
    HEADWAY_BASELINE_STD_MANILA
)

print(
    "Optimized network loaded:",
    optimized_network.fitness_score
)

Headway baseline mean: 12.252840909090908
Headway baseline std: 0.15961257814879687
Optimized network loaded: 0.9081121035755997


In [54]:
# =========================================================
# PREPARE EXISTING NETWORK
# =========================================================

existing_network = Manila_Route_Network

# Use the existing network's own stop-split road graph.
existing_road_graph = graph_of_stops_Manila

# Use the GA's stop-split road graph for optimized routes.
optimized_road_graph = import_networks(
    "Saved Networks/Manila/"
    "stop_graph_Manila.pkl"
)

# Prepare travel times on both graphs.
existing_road_graph = prepare_headway_edge_times(
    existing_road_graph
)

optimized_road_graph = prepare_headway_edge_times(
    optimized_road_graph
)

# Check that all existing intermodal stops have
# identifiable cleaned transportation sources.
for node_id, data in (
    existing_network.graph.nodes(data=True)
):

    if data.get("isTranspo", False):

        if data.get("transport_source_id") is None:
            raise ValueError(
                f"Existing stop {node_id} is missing "
                "transport_source_id."
            )

# Check all route paths against their own road graphs.
for name, network, road_graph in [
    ("Existing", existing_network, existing_road_graph),
    ("Optimized", optimized_network, optimized_road_graph)
]:

    missing_paths = []
    missing_road_edges = []

    for u, v, key, data in (
        network.graph.edges(keys=True, data=True)
    ):

        path = data.get("road_path")

        if not path or len(path) < 2:
            missing_paths.append((u, v, key))
            continue

        for a, b in zip(path[:-1], path[1:]):

            if not road_graph.has_edge(a, b):
                missing_road_edges.append(
                    (key, a, b)
                )

    print(
        name,
        "| missing route paths:",
        len(missing_paths),
        "| missing road edges:",
        len(missing_road_edges)
    )

    assert not missing_paths
    assert not missing_road_edges

print("Manila route-path validation passed.")

Headway edge speed/travel-time attributes prepared.
Headway edge speed/travel-time attributes prepared.
Existing | missing route paths: 0 | missing road edges: 0
Optimized | missing route paths: 0 | missing road edges: 0
Manila route-path validation passed.


In [55]:
# =========================================================
# EXISTING VS OPTIMIZED: SIX FITNESS METRICS
# =========================================================

weights = {
    "weight_coverage": 1.0 / 6.0,
    "weight_connectivity": 1.0 / 6.0,
    "weight_targeted_robustness": 1.0 / 6.0,
    "weight_random_robustness": 1.0 / 6.0,
    "weight_flood_robustness": 1.0 / 6.0,
    "weight_headway_efficiency": 1.0 / 6.0,
}

def evaluate_manila_network(
    network,
    road_graph
):
    score, components = compute_fitness_score(
        network_obj=network,
        graph_of_stops=road_graph,
        zone_population=ZONE_POPULATION_MANILA,
        population_denominator=POPULATION_DENOMINATOR_MANILA,
        total_network_road_distance=TOTAL_NETWORK_ROAD_DISTANCE,
        headway_baseline_mean=HEADWAY_BASELINE_MEAN_MANILA,
        headway_baseline_std=HEADWAY_BASELINE_STD_MANILA,
        **weights
    )

    network.fitness_score = score
    network.fitness_score_components = components

    return score, components


existing_score, existing_components = (
    evaluate_manila_network(
        existing_network,
        existing_road_graph
    )
)

optimized_score, optimized_components = (
    evaluate_manila_network(
        optimized_network,
        optimized_road_graph
    )
)

metric_names = {
    "Coverage": "Coverage Raw",
    "Connectivity": "Connectivity Raw",
    "Targeted Attack Robustness":
        "Targeted Attack Robustness Raw",
    "Random Attack Robustness":
        "Random Attack Robustness Raw",
    "Flood Robustness":
        "Flood Robustness Raw",
    "Headway Efficiency":
        "Headway Efficiency Raw",
    "Average Required Jeepneys":
        "Average Required Jeepneys",
}

comparison_data = {
    label: {
        "Existing": existing_components[key],
        "Optimized": optimized_components[key]
    }
    for label, key in metric_names.items()
}

comparison_data["Overall Fitness"] = {
    "Existing": existing_score,
    "Optimized": optimized_score
}

fitness_comparison_df = pd.DataFrame.from_dict(
    comparison_data,
    orient="index"
)

fitness_comparison_df["Difference"] = (
    fitness_comparison_df["Optimized"]
    - fitness_comparison_df["Existing"]
)

print("=== MANILA FITNESS COMPARISON ===")

display(
    fitness_comparison_df.round(4)
)

# Confirm the optimized fitness agrees with its
# original saved score.
saved_optimized_score = float(
    ga_results["population"][0].fitness_score
)

print(
    "Saved optimized fitness:",
    SAVED_OPTIMIZED_FITNESS
)

print(
    "Recomputed optimized fitness:",
    optimized_score
)

assert np.isclose(
    optimized_score,
    SAVED_OPTIMIZED_FITNESS,
    rtol=1e-8,
    atol=1e-8
), (
    "Recomputed optimized fitness differs from "
    "the saved GA result. Check the shared inputs "
    "and function versions."
)

=== MANILA FITNESS COMPARISON ===


,Existing,Optimized,Difference
Coverage,0.5208,0.9311,0.4104
Connectivity,0.2386,0.6932,0.4545
Targeted Attack Robustness,0.4301,0.9308,0.5006
Random Attack Robustness,0.6669,0.9475,0.2806
Flood Robustness,0.6537,0.9486,0.2948
Headway Efficiency,0.4512,0.9975,0.5463
Average Required Jeepneys,12.2841,11.2955,-0.9886
Overall Fitness,0.4936,0.9081,0.4146


Saved optimized fitness: 0.9081121035755997
Recomputed optimized fitness: 0.9081121035755997


In [56]:
# =========================================================
# SHARED NETWORK SUMMARY FUNCTIONS
# =========================================================

load_ga_function_definitions([
    "get_unique_route_stops",
    "get_route_stop_count",
    "get_route_lengths_km",
    "average_degree",
    "graph_with_distance_weights",
    "weighted_network_diameter_km",
    "avg_betweenness_centrality",
    "avg_closeness_centrality",
    "summarize_network"
])

existing_summary = summarize_network(
    existing_network
)

optimized_summary = summarize_network(
    optimized_network
)

structural_comparison_df = pd.DataFrame({
    "Existing": existing_summary,
    "Optimized": optimized_summary
})

print("=== MANILA NETWORK COMPARISON ===")

display(
    structural_comparison_df.round(4)
)

Loaded GA functions: get_unique_route_stops, get_route_stop_count, get_route_lengths_km, average_degree, graph_with_distance_weights, weighted_network_diameter_km, avg_betweenness_centrality, avg_closeness_centrality, summarize_network
=== MANILA NETWORK COMPARISON ===


,Existing,Optimized
Coverage Score,0.5208,0.9311
Connectivity Score,0.2386,0.6932
Targeted-Attack Robustness Score,0.4301,0.9308
Random-Attack Robustness Score,0.6669,0.9475
Flood Robustness Score,0.6537,0.9486
Headway Efficiency Score,0.4512,0.9975
Average Required Jeepneys,12.2841,11.2955
Total Fitness Score,0.4936,0.9081
Total Network Nodes,396.0000,802.0000
TotalRoutes,88.0000,88.0000


In [57]:
# =========================================================
# DIRECTIONAL ROUTE STATISTICS
# =========================================================

def directional_route_summary(network):

    forward = [
        float(route.length)
        for route in network.routes
        if route is not None
    ]

    reverse = [
        float(route.length)
        for route in network.reversed_routes
        if route is not None
    ]

    combined = forward + reverse

    result = {
        "Forward Routes": len(forward),
        "Reverse Routes": len(reverse),
        "Total Directional Routes": len(combined),
        "Total Route Distance (km)": np.sum(combined),
        "Mean Route Distance (km)":
            np.mean(combined) if combined else np.nan,
        "Median Route Distance (km)":
            np.median(combined) if combined else np.nan,
        "Min Route Distance (km)":
            np.min(combined) if combined else np.nan,
        "Max Route Distance (km)":
            np.max(combined) if combined else np.nan,
        "Route Distance Std (km)":
            np.std(combined, ddof=0)
            if combined else np.nan
    }

    return result


directional_comparison_df = pd.DataFrame({
    "Existing": directional_route_summary(
        existing_network
    ),
    "Optimized": directional_route_summary(
        optimized_network
    )
})

display(
    directional_comparison_df.round(4)
)

,Existing,Optimized
Forward Routes,88.0000,88.0000
Reverse Routes,86.0000,88.0000
Total Directional Routes,174.0000,176.0000
Total Route Distance (km),2370.6111,2251.7151
Mean Route Distance (km),13.6242,12.7938
Median Route Distance (km),12.2730,12.9555
Min Route Distance (km),0.8437,7.7870
Max Route Distance (km),34.5116,14.9730
Route Distance Std (km),7.5841,1.6660


In [58]:
def additional_structural_metrics(network):

    graph = network.graph
    undirected = nx.Graph(graph)

    betweenness = nx.betweenness_centrality(
        undirected,
        normalized=True
    )

    closeness = nx.closeness_centrality(
        undirected
    )

    largest_component = max(
        (
            len(component)
            for component in nx.weakly_connected_components(
                graph
            )
        ),
        default=0
    )

    return {
        "Total Unique Stops": graph.number_of_nodes(),
        "Largest Weakly Connected Component":
            largest_component,
        "Maximum Betweenness Centrality":
            max(betweenness.values(), default=0),
        "Maximum Closeness Centrality":
            max(closeness.values(), default=0)
    }


additional_comparison_df = pd.DataFrame({
    "Existing": additional_structural_metrics(
        existing_network
    ),
    "Optimized": additional_structural_metrics(
        optimized_network
    )
})

display(additional_comparison_df.round(4))

,Existing,Optimized
Total Unique Stops,396.0000,802.0000
Largest Weakly Connected Component,292.0000,802.0000
Maximum Betweenness Centrality,0.2235,0.1382
Maximum Closeness Centrality,0.1307,0.1329


In [59]:
export_networks(
    existing_network,
    f"{pikl_filepath}"
    "Manila Route Network Thesis Baseline Evaluated.pikl"
)

print(
    "Saved evaluated Manila baseline.",
    "Fitness:",
    existing_network.fitness_score
)

Saved evaluated Manila baseline. Fitness: 0.4935612371663729


### Export CSV

In [37]:
with open('Original Jeepney Routes/Makati_routes.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['lat', 'long', 'route id'])
    
    for route in Makati_Route_Network.routes:
        for connection in route.route:
            if route.route.index(connection) == 0:
                lat = graph_of_stops_Makati.nodes[connection[0]]['y']
                long = graph_of_stops_Makati.nodes[connection[0]]['x']
                writer.writerow([lat, long, route.route_id])
            
            lat = graph_of_stops_Makati.nodes[connection[-1]]['y']
            long = graph_of_stops_Makati.nodes[connection[-1]]['x']
            writer.writerow([lat, long, route.route_id])

print("Data exported successfully.")

Data exported successfully.


In [38]:
with open('Original Jeepney Routes/Mandaluyong_routes.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['lat', 'long', 'route id'])
    
    for route in Mandaluyong_Route_Network.routes:
        for connection in route.route:
            if route.route.index(connection) == 0:
                lat = graph_of_stops_Mandaluyong.nodes[connection[0]]['y']
                long = graph_of_stops_Mandaluyong.nodes[connection[0]]['x']
                writer.writerow([lat, long, route.route_id])
            
            lat = graph_of_stops_Mandaluyong.nodes[connection[-1]]['y']
            long = graph_of_stops_Mandaluyong.nodes[connection[-1]]['x']
            writer.writerow([lat, long, route.route_id])

print("Data exported successfully.")

Data exported successfully.
